# The Plate Language

Connor Hanley (Indiana University)

## Introduction

Thought seems to be structured. How can networks of connected units like neurons have structure? McCulloch and Pitts demonstrated that, under certain assumptions, artificial neural networks are equivalent to a simple propositional logic \[@mccullochPitts1942\]. So too has it been demonstrated that neural networks can exhibit the same systematic, compositional, and generative relations that psychological states are taken to have \[see @fodorConnectionismCognitiveArchitecture1988; @smolenskyTensorProductVariable1990; and also @plateHolographicReducedRepresentations1995; @gaylerVectorSymbolicArchitectures2004\]. Likewise, programming languages like the linearly typed lambda calculus \[@velez-ginorioCompilingLinearNeurons2026; and @velez-ginorioCompilingRecurrentNeurons2026\]

In this work we will describe an interpreter for the LISP programming language which constructs and runs an equivalent network based on Vector Symbolic Architectures (VSAs) \[@tomkins-flanaganHeyPenttiWe2025; @hanleyHeyPenttiWe2025\]. VSAs are a model of computation using high-dimensional vectors that support compositional, systematic, and generative patterns \[@smolenskyTensorProductVariable1990; @plateHolographicReducedRepresentations1995; @kanervaHyperdimensionalComputingIntroduction2009; @gaylerVectorSymbolicArchitectures2004\]. We will demonstrate how VSAs are able to encode to (1) a subset of the LISP language, called LISP$_\text{Plate}$ and (2) be ordered in a way so as to reproduce the semantics of a LISP$_\text{Plate}$.

To reach this point, first we will discuss VSAs in @sec-vsa, in particular, Holographic Reduced Representations \[HRRS, @plateHolographicReducedRepresentations1995\] and Time-domain Residue Hyperdimensional Computing \[TRHC, @kymnComputingResidueNumbers2024\]. Afterwards, we will describe a subset of the LISP language in @sec-lang, which we will provide a mapping for to VSAs in @sec-enc. Finally, we will describe how the mapped VSAs can be procedurally manipulated to reproduce the semantics of LISP in @sec-interp.

## Vector Symbolic Architectures

VSA/HDC is a implementation of computation in high-dimensional vectors \[or tensors, see @smolenskyTensorProductVariable1990\] that provide neural networks with the ability to construct and manipulate *distributed* patterns that are compositional, systematic, and generative. Will provide a general definition of VSAs, then focus in on the two VSAs that we will use to explore list encodings: Holographic Reduced Representations \[HRRs, from @plateHolographicReducedRepresentations1995\] and Time-domain Residue Hyperdimensional Computing \[TRHC, counterpart to RHC from @kymnComputingResidueNumbers2024; cf. @voelkerSimulatingPredictingDynamical2021\].

A *Vector Symbolic Architecture* $\mathcal{V}$ is a structure $\langle X^{D_1 \times \dots \times D_n}, \sim,  \otimes, \oplus \rangle$, where:

1.  A *pattern set* $X^{D_1 \times \dots \times D_n}$,
2.  There is a *similarity* function $$
    x_1 \sim x_2 : X^{D_1 \times \dots \times D_n} \times X^{D_1 \times \dots \times D_n} \to [0, 1];
    $$
3.  There is a *binding* function, $$
    x_1 \otimes x_2 : X^{D_1 \times \dots \times D_n} \times X^{D_1 \times \dots \times D_n} \to X^{D_1 \times \dots \times D_n},
    $$ such that $x_1 \otimes x_2$ is neither similar to $x_1$ nor $x_2$, and the binding operation admits an approximate inverse, $$
    \begin{align*}
     [x_1 \otimes^{-1} (x_1 \otimes x_2) \sim x_2] &\approx 1, \\
     [x_2 \otimes^{-1} (x_1 \otimes x_2) \sim x_1] &\approx 1;
    \end{align*}
    $$
4.  And, there is a *bundling* or *superposition* operation, $$
    x_1 \oplus x_2 : X^{D_1 \times \dots \times D_n} \times X^{D_1 \times \dots \times D_n} \to X^{D_1 \times \dots \times D_n},
    $$ such that $x_1 \oplus x_2$ is similar to both $x_1$ and $x_2$.

Some VSAs include an extra *permutation* operation that functions like a one-place binding, such that $$
\Pi(x) : X^{D_1 \times \dots \times D_n} \to X^{D_1 \times \dots \times D_n},
$$ and, $$
\Pi(x) \sim x \approx 0, \quad \Pi^{-1} \Pi (x) \sim x \approx 1.
$$ We will not include it in our definition, but we will use the permutation operation in the following.

@def-vsa can be operationalized by the following abstract class definition:

``` python
from abc import ABCMeta, abstractmethod
from typing import ClassVar, Self

import numpy as np
import numpy.typing as npt

class VSA[T: np.generic](metaclass=ABCMeta):
    """Abstract base class of all VSA implementations.

    All VSAs must implement the following methods:
    """

    data: npt.NDArray[T]
    dtype: ClassVar[type[np.generic]]

    @staticmethod
    @abstractmethod
    def bind(x: npt.NDArray[T], y: npt.NDArray[T]) -> npt.NDArray[T]:
        """Vector symbolic binding."""
        ...

    @staticmethod
    @abstractmethod
    def bundle(x: npt.NDArray[T], y: npt.NDArray[T]) -> npt.NDArray[T]:
        """Vector symbolic bundling."""
        ...

    @staticmethod
    @abstractmethod
    def unbind(x: npt.NDArray[T], y: npt.NDArray[T]) -> npt.NDArray[T]:
        """Vector symbolic unbinding."""
        ...

    @staticmethod
    @abstractmethod
    def similarity(x: npt.NDArray[T], y: npt.NDArray[T]) -> float:
        """Vector symbolic similarity."""
        ...

    @classmethod
    @abstractmethod
    def new(cls, dim: int) -> Self:
        """Initialize a new vector."""
        ...

    @classmethod
    @abstractmethod
    def from_array(cls, array: npt.NDArray[T]) -> Self:
        """Create a VSA from an array."""
        ...

    @abstractmethod
    def __hash__(self) -> int: ...
```

### Holographic Reduced Representations

Holographic Reduced Representations \[HRRs, @plateHolographicReducedRepresentations1995\] are a VSA defined over $D$-dimensional real vectors, which uses cosine similarity, element-wise multiplication in the Fourier domain, and element-wise summation to implement the VSA operations in @def-vsa.

*Holographic Reduced Representations* (HRRs) are a VSA $\mathcal{H}$, where

1.  The set of high-dimensional patterns is $\mathbb{R}^D$;
2.  The similarity function is *cosine similarity*: $$
     x_1 \sim x_2 = \frac{x_1^\top x_2}{|x_1|_2 |x_2|_2};
    $$
3.  Binding is *circular convolution*: $$
     x_1 \otimes x_2 = \mathcal{F}^{-1}\left[ \mathcal{F}(x_1) \odot \mathcal{F}(x_2)  \right],
    $$ and unbinding *circular correlation*: $$
    x_1 \otimes^{-1} x_2 = \mathcal{F}^{-1} \left[ \mathcal{F}(x_1) \odot \overline{\mathcal{F}(x_2)} \right];
    $$
4.  And, bundling is the element-wise sum of $x_1$ and $x_2$: $$
     x_1 \oplus x_2 = x_1 + x_2.
    $$

Any vector in the pattern set $\mathbb{R}^D$ of @def-hrr is able to have the operations of (2-4) performed on it; however, in order to make the use of the HRR’s principled, assume in the following that newly generated HRR vectors are unitary[1] Demonstrating that HRRs are VSAs aside from stipulation is trivial, so we will leave it out of our consideration here.

@def-hrr can be operationalized as:

[1] This is to aid in circular correlation.

``` python
import math
from typing import Literal, Self, cast, override

import numpy as np
import numpy.typing as npt
from numpy.fft import fft, ifft, irfft, rfft


type ArrayF64 = npt.NDArray[np.float64]

type Scheme = Literal["unitary", "gaussian"]
SCHEMES: tuple[Scheme, ...] = ("unitary", "gaussian")
"""How a fresh vector-symbol is drawn. Unitary vectors have a flat magnitude
spectrum, which makes `HRR.inv` their exact inverse rather than an approximate
one; Gaussian vectors are the classical choice.
"""


class HRR(VSA[np.float64]):
    """Holographic reduced representation vectors.

    The vectors of HRR are sampled from a normal distribution. Implements
    binding through circular convolution.
    """

    data: ArrayF64
    dtype = np.float64

    def __init__(self, data: ArrayF64) -> None:
        self.data = data

    @classmethod
    def normal(cls, size: int, sd: float | None = None) -> Self:
        """Create a new HRR by sampling from the normal distribution.

        Args:
            size (int): The dimensionality of the new HRR vector-symbol.
            sd (float | None): Defaults to `None`, the standard deviation
            of the normal distribution.

        Returns:
            A new HRR vector-symbol.
        """
        if sd is None:
            sd = 1.0 / math.sqrt(size)
        data = np.random.normal(scale=sd, size=size)
        data /= np.linalg.norm(data)
        return cls(data)

    @classmethod
    def unitary(cls, size: int) -> Self:
        """Create a new HRR whose magnitude spectrum is flat.

        Every frequency has a magnitude of one, so binding neither amplifies
        nor attenuates any of them: `HRR.inv` is then the exact inverse of
        binding rather than an approximate one, and the norm survives any
        number of bindings.

        Args:
            size (int): The dimensionality of the new HRR vector-symbol.

        Returns:
            A new unitary HRR vector-symbol.
        """
        spectrum = rfft(np.random.normal(size=size))
        return cls(irfft(spectrum / np.abs(spectrum), n=size))

    @override
    @classmethod
    def from_array(cls, array: ArrayF64) -> Self:
        """Create a new HRR from an array.

        Args:
            x (npt.NDArray[np.float64]): A raw float array.

        Returns:
            A new HRR vector-symbol, drawn from `x`.
        """
        return cls(array)

    @override
    @classmethod
    def new(cls, dim: int, scheme: Scheme = "unitary") -> Self:
        """Create a new vector-symbol.

        Args:
            dim (int): The dimensionality of the new vector-symbol.
            scheme (Scheme): Defaults to `"unitary"`, how the vector-symbol is
                drawn. See `SCHEMES`.

        Returns:
            A new HRR vector-symbol.

        Raises:
        -   ValueError: If `scheme` is not one of `SCHEMES`.
        """
        # Checked at runtime because the scheme reaches here as a string from
        # the command line, where the type is not enforced.
        if scheme not in SCHEMES:
            raise ValueError(f"scheme must be one of {SCHEMES}, got {scheme!r}")

        return cls.unitary(dim) if scheme == "unitary" else cls.normal(dim)

    @override
    @staticmethod
    def bind(x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """The product operation in the HRR VSA.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.
            y (npt.NDArray[np.float64]): The right-hand side of the operation.

        Returns:
            The circular convolution of `x` and `y`. Here, it is implemented
            through the fast Fourier transform.
        """
        return cast(ArrayF64, ifft(fft(x) * fft(y)).real)

    @override
    @staticmethod
    def bundle(x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """The HRR VSA sum operation.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.
            y (npt.NDArray[np.float64]): The right-hand side of the operation.

        Returns:
            The element-wise sum of the left-hand side and the right-hand side.
        """
        return x + y

    @staticmethod
    def inv(x: ArrayF64) -> ArrayF64:
        """The approximate inverse for HRR.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.

        Returns:
            The approximate inverse of `x`.
        """

        return x[np.r_[0, x.size - 1 : 0 : -1]]

    @override
    @staticmethod
    def unbind(x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """The unbinding operation in the HRR VSA.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.
            y (npt.NDArray[np.float64]): The right-hand side of the operation.

        Returns:
            The binding of the left-hand side with the approximate inverse
            of the right hand side.
        """
        return HRR.bind(x, HRR.inv(y))

    @override
    @staticmethod
    def similarity(x: ArrayF64, y: ArrayF64) -> float:
        """Approximated kernel for HRR. Measures the 'distance' between
        the left-hand and right-hand side.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.
            y (npt.NDArray[np.float64]): The right-hand side of the operation.

        Returns:
            The 'distance' between the left-hand side and the right-hand
            side, a value between -1 and 1.
        """
        mag = float(np.linalg.norm(x) * np.linalg.norm(y))
        if mag == 0.0:
            return 0.0
        else:
            return float(np.dot(x, y) / mag)
```

In [3]:
from abc import update_abstractmethods


def HRRadd__(self, rhs: HRR | float) -> Self:
    """See `HRR.bundle`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.bundle(self.data, rhs.data))
    else:
        return cls(self.data + rhs)

def HRRradd__(self, rhs: HRR | float) -> Self:
    """See `HRR.bundle`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.bundle(self.data, rhs.data))
    else:
        return cls(self.data + rhs)

def HRRsub__(self, rhs: HRR | float) -> Self:
    """Element-wise subtraction."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(self.data - rhs.data)
    else:
        return cls(self.data - rhs)

def HRRmul__(self, rhs: HRR | float) -> Self:
    """Scalar multiplication or `HRR.bind`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.bind(self.data, rhs.data))
    else:
        return cls(self.data * rhs)

def HRRrmul__(self, rhs: HRR | float) -> Self:
    """Scalar multiplication or `HRR.bind`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.bind(self.data, rhs.data))
    else:
        return cls(self.data * rhs)

def HRRtruediv__(self, rhs: HRR | float) -> Self:
    """Scalar division or `HRR.unbind`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.unbind(self.data, rhs.data))
    elif isinstance(rhs, int):
        return cls(self.data / rhs)
    else:
        return cls((self.data / rhs).astype(np.float64))

def HRRinvert__(self) -> Self:
    """See `HRR.inv`."""
    cls = type(self)
    return cls(cls.inv(self.data))

def HRRneg__(self) -> Self:
    """Element-wise negation."""
    return type(self)(-self.data)

def magnitude(self) -> float:
    """The magnitude of the raw vector."""
    return math.sqrt(self.data @ self.data) / self.data.size

def HRRmatmul__(self, other: HRR | ArrayF64) -> float | ArrayF64:
    """Matrix multiplication."""
    if isinstance(other, HRR):
        return self.data @ other.data
    else:
        if len(other.shape) == 2:
            return (self.data @ other).astype(np.float64)
        else:
            return self.data @ other

def sim(self, other: HRR | ArrayF64) -> float:
    """See `HRR.similarity`."""
    if isinstance(other, HRR):
        return HRR.similarity(self.data, other.data)
    else:
        return HRR.similarity(self.data, other)

@override
def HRRstr__(self) -> str:
    return f"{type(self).__name__}({self.data})"

@override
def HRRhash__(self) -> int:
    return hash(self.data.tobytes())


for _method in (
    HRRadd__,
    HRRradd__,
    HRRsub__,
    HRRmul__,
    HRRrmul__,
    HRRtruediv__,
    HRRinvert__,
    HRRneg__,
    magnitude,
    HRRmatmul__,
    sim,
    HRRstr__,
    HRRhash__,
):
    name = str(_method.__name__).replace("HRR", "__")
    print(name)
    setattr(HRR, name, _method)

# `VSA.__hash__` is abstract, so recompute HRR's abstract methods now that
# it has been implemented.
_ = update_abstractmethods(HRR)

__add__
__radd__
__sub__
__mul__
__rmul__
__truediv__
__invert__
__neg__
magnitude
__matmul__
sim
__str__
__hash__

### Time-domain Residue Hyperdimensional Computing

Time-domain Residue Hyperdimensional Computing (TRHC) is a modification of the Residue Hyperdimensional Computing VSA \[RHC, @kymnComputingResidueNumbers2024\] which uses a residue number encoding to represent natural numbers and simple arithmetic in VSAs. Strictly speaking, according to @def-vsa TRHC (and RHC) is not one single VSA but two distinct VSAs, since TRHC defines two binding operations that perform addition and multiplication over their encoded residue number. For our purposes, we will focus only on the variant with additive binding since the multiplicative binding operation is tedious to deal with.

Unlike HRRs, TRHC requires some preliminary definitions. Namely, since TRHC was developed to represent numbers using a residue encoding, we have to first define what that means.

TRHC vectors must have the following properties:

1.  TRHC vectors must have a unit norm, such that $\|x\| = 1$; and,
2.  TRHC vectors must be *unitary under binding*, meaning that binding (circular convolution) and unbinding (circular correlation) must be exact inverses.

Suppose we have a set of odd moduli $m_1, m_2, \dots, m_k$. For each modulus, let $b_i$ denote the *base vector* of modulus $m_i$ of dimension $D$. A base vector is constructed by[1]:

1.  If $D$ is even, then:
    -   $b_i[0] = 0$,
    -   $b_i[1, \dots, D / 2 - 1]$ are sampled from $\{0, \dots, m_i - 1\}$
    -   $b_i[D / 2] = 0$, which is the *Nyquist* bin; and,
    -   $b_i[D/2 + 1, \dots, D]$ is the mirror and flipped sign of $b_i[1, \dots, D/2-1]$.
2.  If $D$ is odd, then:
    -   $b_i[0] = 0$,
    -   $b_i[1, \dots, \lfloor D / 2\rfloor]$ are sampled from $\{0, \dots, m_i - 1\}$, and
    -   $b_i[\lfloor D / 2\rfloor + 1, \dots, D]$ are mirror and flipped sign of $b_i[1, \dots, \lfloor D/2 \rfloor]$.

To get the residue effect, we multiply by $2 \pi / m_i$ and exponentiate, and then apply the inverse Fourier transform to return to the time domain: $$
b_i = \exp\left( \frac{2 \pi b_i}{m_i} \right)
$$ For all base vectors, let $b_i^n$ denote: $$
b_i^n = \mathcal{F}^{-1} \left[ \mathcal{F}(b_i)^n \right].
$$ Then, the encoding of a natural number $n$ in TRHC with moduli $m_1, m_2, \dots, m_k$ is: $$
v(n) = b_1^n \otimes b_2^n \otimes \dots \otimes b_k^n,
$$ where $\cdot \otimes \cdot$ denotes circular convolution. This gives us the residue result, such that each vector stores the residues $n \bmod m_1, \dots, n \bmod m_k$, and by the Chinese Remainder Theorem, $n$ is uniquely determined modulo $\prod_i m_i$.

Additive *Time-domain Residue Hyperdimensional Computing*[2] (aTRHC) is a VSA with:

1.  A set of $D$-dimensional real vectors, $\mathbb{R}^D$;
2.  Similarity is *cosine similarity*;
3.  Binding via circular convolution, unbinding via circular correlation; and,
4.  Bundling via element-wise addition.

aTRHC is said to be *additive* because circular convolution obeys the following property: $$
v(n) \otimes v(m) = v\left(n + m \bmod \prod_i m_i\right).
$$

@def-trhc can be operationalized as:

[1] We use $b_i[\dots]$ to denote indexing into the vector, following `python` style, so as to not confuse the index into the vector with the index matching the modulus.

[2] *Multiplicative* Binding is described for the Fourier-domain in @kymnComputingResidueNumbers2024.

``` python
from typing import ClassVar, Self, override

import numpy as np
import numpy.typing as npt


__all__ = ["TRHC"]

type ArrayF64 = npt.NDArray[np.float64]


class TRHC(HRR):
    """Time-domain Residue Hyperdimensional Computing.

    So-called, because it deals with RHC in the time-domain, as opposed to the frequency
    domain.
    """

    data: ArrayF64
    moduli: ClassVar[list[int]] = [3, 5, 7, 11]
    basis: ClassVar[list[ArrayF64]] = []

    @override
    @classmethod
    def new(cls, dim: int, scheme: Scheme = "unitary") -> Self:
        """Create a new vector-symbol.

        Args:
            dim (int): The dimensionality of the new vector-symbol.
            scheme (Scheme): Ignored. A Gaussian vector is not unitary, and so
                would leave the residue cycle as soon as it was bound to
                itself. The argument is kept only to match `HRR.new`.

        Returns:
            A new unitary TRHC vector-symbol.
        """
        return cls.unitary(dim)

    @staticmethod
    def generate_base_vector(
        rng: np.random.Generator, modulus: int, dim: int
    ) -> ArrayF64:
        """Generates an RHC base vector in the time domain.

        Args:
        -   rng (np.random.Generator): The random number generator.
        -   modulus (int): The modulus of the base vector.
        -   dim (int): The dimension of the base vector.

        Returns:
            An TRHC base vector in the time domain.
        """

        # Only the non-negative frequencies are drawn; `irfft` mirrors them
        # into the conjugate-symmetric half, which keeps the result real.
        k_choices = np.zeros(dim // 2 + 1, dtype=int)
        k_choices[0] = 0
        k_choices[1:] = rng.choice(modulus, dim // 2)

        if dim % 2 == 0:
            # The Nyquist bin is its own conjugate, so its phase must be 0 or
            # pi. Odd moduli admit no phase of pi, leaving 0 as the only choice.
            k_choices[-1] = 0

        phases = 2 * np.pi * k_choices / modulus
        return np.fft.irfft(np.exp(1j * phases), n=dim)

    @classmethod
    def generate_basis(cls, dim: int) -> None:
        """Generate a fresh basis, one base vector per modulus.

        Args:
        -   dim (int): The dimension of the base vectors.
        """
        rng = np.random.default_rng()
        cls.basis = [cls.generate_base_vector(rng, mod, dim) for mod in cls.moduli]

    @staticmethod
    def _bind_power(vec: ArrayF64, num: int) -> ArrayF64:
        """Raise a base vector to an integer binding power.

        Equivalent to binding `vec` with itself `num` times, but evaluated in the
        frequency domain so the result stays unitary for any `num`. A power of 0
        gives the binding identity.

        Args:
        -   vec (ArrayF64): The base vector.
        -   num (int): The binding power.

        Returns:
            The base vector raised to the given binding power.
        """
        return np.fft.ifft(np.fft.fft(vec) ** num).real

    @classmethod
    def number(
        cls,
        num: int,
        dim: int,
        alternative_basis: list[ArrayF64] | None = None,
    ) -> Self:
        """Create an TRHC vector from a number.

        Args:
        -   num (int): The number to convert.
        -   dim (int): The dimension of the vector.
        -   alternative_basis (list[ArrayF64] | None): An optional alternative basis to use.

        Returns:
            An TRHC vector representing the number.

        Raises:
        -   ValueError: If `alternative_basis` is provided and is empty.
        -   ValueError: If the basis dimension does not match the dimension of the vector.
        """

        if not cls.basis:
            cls.generate_basis(dim)

        basis = cls.basis

        if alternative_basis is not None and len(alternative_basis) == 0:
            raise ValueError("alternative_basis must not be empty")
        elif alternative_basis is not None and isinstance(
            alternative_basis[0], np.ndarray
        ):
            basis = alternative_basis

        if basis[0].shape[0] != dim:
            raise ValueError("basis dimension must match dim")

        rhc_num = cls._bind_power(basis[0], num)
        for i in range(1, len(basis)):
            rhc_num = cls.bind(rhc_num, cls._bind_power(basis[i], num))

        return cls(rhc_num)

    @classmethod
    def residue_add(cls, x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """Perform TRHC arithmetical addition.

        Args:
        -   x (ArrayF64): The first vector.
        -   y (ArrayF64): The second vector.

        Returns:
            The result of the addition.
        """
        return cls.bind(x, y)

    @classmethod
    def residue_sub(cls, x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """Perform TRHC arithmetical subtraction.

        Args:
        -   x (ArrayF64): The first vector.
        -   y (ArrayF64): The second vector.

        Returns:
            The result of the subtraction.
        """
        return cls.unbind(x, y)
```

aTRHC will be important later on for us when we deal with numerical representations in @sec-numbers, where we will compare its abilities to an alternative Peano encoding.

## The Language

VSAs exhibit compositional and systematic relations in virtue of operations (2-4) in @def-vsa. They can also be used to encode data structures, such as lists, graphs, and trees \[@kleykoSurveyHyperdimensionalComputing2023\]. Following @tomkins-flanaganHeyPenttiWe2025 and @hanleyHeyPenttiWe2025, we will instead encode and interpret a full-fledged programming language. In particular, we will be encoding a subset of the R5RS Scheme standard \[@kelseyRevisedReport1998\].

### What is a Language

More formally speaking, a language $\mathcal{L}$ is a set of strings that are closed under a recursive definition of set membership. We will call the sentences that must hold true of a string the *formation rules* of $\mathcal{L}$. Likewise, a string which obeys these formation rules will be called a *well-formed formula of* $\mathcal{L}$.

A *language* $\mathcal{L}$ is a set of strings defined by a set of *formation rules* $S_1, S_2, \dots, S_n$, such that strings are only in the set iff it is true of them that $S_1, S_2, \dots, S_n$. A string that is in the language $\mathcal{L}$ is said to be a *well-formed formula of* $\mathcal{L}$.

To give a toy example, let us discuss the lanaguage $\mathcal{L}_\text{fruit}$.

Let $\mathcal{A}$ be the set of atomic symbols $\{\text{apple}, \text{banana}, \text{pear}\}$. Then, the language $\mathcal{L}_\text{fruit}$ is given by the following formation rules:

1.  (Atomic formulae) If $a \in \mathcal{A}$, then $a \in \mathcal{L}_\text{fruit}$
2.  (Compound formulae) If $x_1, x_2 \in \mathcal{L}_\text{fruit}$, then the *disjunction* of $x_1$ and $x_2$, $(x_1 \lor x_2) \in \mathcal{L}_\text{fruit}$ and the *conjunction* of $x_1$ and $x_2$, $(x_2 \land x_2) \in \mathcal{L}_\text{fruit}$.
3.  (Closure Under the Formation Rules) No other string is in $\mathcal{L}_\text{fruit}$.

This definition gives us simple sentences like $\text{apple}$, but also complex sentences like $((\text{apple} \lor (\text{banana} \land \text{pear})) \land (\text{banana} \lor \text{apple}))$. Not a very useful language except for maybe expressing personal preferences[1], however the coupling of the language definition with formation rules allows for us to express the language even more tersely. Instead of providing an explicit definition of formation rules, we can provide a *grammar* of a language $\mathcal{L}$ by writing it out in Backus-Naur Form.

A *grammar* of a language $\mathcal{L}$ is a $4$-tuple $G = \langle N, \Sigma, P, S, \rangle$, where:

1.  $N$ is a finite set of *non-terminals*, written $\langle \dots \rangle$;
2.  $\Sigma$ isa a finite set of *terminals*, the atomic symbols of the language, where $N \cap \Sigma = \varnothing$;
3.  $P$ is a finite set of *productions*, each with the form $A ::= \alpha$, for $A \in N$ and $\alpha \in (N \cup \Sigma)^*$; and,
4.  $S \in N$ is a *start symbol*.

We write $V = N \cup \Sigma$ and let $\varepsilon$ denote the empty string. The *Backus-Naur Form* (BNF) is a notation for $P$ in which several productions sharing a left-hand side are collapsed into a single rule by a bar: the rule $A ::= \alpha_1 \mid \alpha_2 \mid \dots \mid \alpha_n$ abbreviate the $n$ productions $A ::= \alpha_i$.

$G$ *derives* $\gamma$ from $\beta$ in one step, denoted by $\beta \Rightarrow \gamma$, when $\beta = \mu A \nu$ and $\gamma = \alpha$ in $P$; i.e., when $\gamma$ is the result of replacing one nonterminal occurrence in $\beta$ by the right-hand side of one of its rules. Let $\Rightarrow^*$ denote the reflexive transitive closure of $\Rightarrow$. Then, the language generated by $G$ is: $$
\mathcal{L}(G) = \{w \in \Sigma^* : S \Rightarrow^* w\}.
$$

Since $\mathcal{L}(G)$ in @def-bnf is defined for exactly the terminal strings derivable from $S$, a grammar carries the clsoure property clause ((3) of @def-lang-fruit) for free.

The grammar of $\mathcal{L}_\text{fruit}$ is given by:

1.  The set of non-terminals $N$: $$
    N = \{ \langle \mathbf{atomic} \rangle, \langle \mathbf{disjunction} \rangle, \langle \mathbf{conjunction} \rangle, \langle \textbf{expression} \rangle  \}
    $$

2.  The set of terminal symbols $\Sigma$: $$
    \Sigma = \{ \text{apple}, \text{banana}, \text{pear}, \land, \lor \}
    $$

3.  The set of productions $P$: $$
    \begin{align*}
    P =  \{&\langle \mathbf{atomic} \rangle ::= \text{apple} \mid \text{banana} \mid \text{pear}, \\
     &\langle \mathbf{conjunction} \rangle ::= \langle \textbf{expression} \rangle \land \langle \textbf{expression}, \rangle \\
     &\langle \mathbf{disjunction} \rangle ::= \langle \textbf{expression} \rangle \lor \langle \textbf{expression}, \rangle \\
     &\langle \mathbf{expression} \rangle ::= \langle \mathbf{atomic} \rangle \mid \langle \mathbf{disjunction} \rangle \mid \langle \mathbf{conjunction} \rangle
    \}
    \end{align*}
    $$

4.  The start symbol $S$ is $\langle \mathbf{expression} \rangle$.

The BNF of @def-fruit-bnf can be sufficiently inferred merely from the set of production rules: $$
\begin{align*}
    \langle \mathbf{atomic} \rangle &::= \text{apple} \mid \text{banana} \mid \text{pear}, \\
    \langle \mathbf{conjunction} \rangle &::= \langle \textbf{expression} \rangle \land \langle \textbf{expression}, \rangle \\
    \langle \mathbf{disjunction} \rangle &::= \langle \textbf{expression} \rangle \lor \langle \textbf{expression}, \rangle \\
    \langle \mathbf{expression} \rangle &::= \langle \mathbf{atomic} \rangle \mid \langle \mathbf{disjunction} \rangle \mid \langle \mathbf{conjunction} \rangle
\end{align*}
$$

BNF grammars already conveniently give a way to easily express the grammar of a language using algebraic data types, or tagged unions. Our grammar for $\mathcal{L}_\text{fruit}$ can be operationalized as follows:

[1] Though, it should be clear that one could define really any relevant language in the provided manner. For example, propositional logic is not too far off from our definition of $\mathcal{L}_\text{fruit}$.

``` python
from dataclasses import dataclass
from typing import Literal

@dataclass
class Atom:
    x: Literal["apple", "banana", "pear"]


@dataclass
class Conjunction:
    lhs: "Expression"
    rhs: "Expression"


@dataclass
class Disjunction:
    lhs: "Expression"
    rhs: "Expression"


type Expression = Atom | Conjunction | Disjunction
```

Note, in @lst-fruit we ignore the conjunction and disjunction symbols. This purely for the sake of convenience: the fact that we separate the non-terminal productions gives us sufficient information about the form so as to distinguish them. If we modified the production rules to have a single $\langle \mathbf{compound} \rangle$ production rule, then we would have to keep the disjunction and conjunction symbols in our operationalization.

### The Grammar of LISP$_\text{Plate}$

Following @def-fruit-bnf, we give the grammar of LISP$_\text{Plate}$ by its production rules alone. The rules fall into four groups: a *lexical* grammar that fixes the tokens, a *reader* grammar that turns a sequence of tokens into data, an *evaluated* subset that picks out the data which are also programs, and a small grammar of quasiquotation templates. Both the reader and the evaluated subset have a start symbol for whole programs; we distinguish them by writing $\langle \mathbf{program\text{-}reader} \rangle$ and $\langle \mathbf{program\text{-}parser} \rangle$ respectively.

The lexical grammar assumes that the input has been lower-cased and that whitespace between tokens is discarded: $$
\begin{align*}
    \langle \mathbf{token} \rangle &::= \texttt{(} \mid \texttt{)} \mid \texttt{'} \mid \texttt{`} \mid \texttt{,} \mid \texttt{.} \mid \langle \mathbf{operator} \rangle \\
        &\qquad \mid \langle \mathbf{boolean} \rangle \mid \langle \mathbf{integer} \rangle \mid \langle \mathbf{word} \rangle \\
    \langle \mathbf{operator} \rangle &::= \texttt{+} \mid \texttt{-} \mid \texttt{*} \mid \texttt{/} \\
    \langle \mathbf{boolean} \rangle &::= \texttt{\#t} \mid \texttt{\#f} \\
    \langle \mathbf{integer} \rangle &::= \langle \mathbf{digit} \rangle \mid \langle \mathbf{digit} \rangle \langle \mathbf{integer} \rangle \\
    \langle \mathbf{word} \rangle &::= \langle \mathbf{letter} \rangle \langle \mathbf{word\text{-}tail} \rangle \\
    \langle \mathbf{word\text{-}tail} \rangle &::= \varepsilon \mid \langle \mathbf{letter} \rangle \langle \mathbf{word\text{-}tail} \rangle \mid \texttt{?} \langle \mathbf{word\text{-}tail} \rangle \\
    \langle \mathbf{letter} \rangle &::= \texttt{a} \mid \texttt{b} \mid \dots \mid \texttt{z} \\
    \langle \mathbf{digit} \rangle &::= \texttt{0} \mid \texttt{1} \mid \dots \mid \texttt{9} \\
    \langle \mathbf{keyword} \rangle &::= \texttt{define} \mid \texttt{lambda} \mid \texttt{if} \mid \texttt{let} \mid \texttt{and} \mid \texttt{or} \mid \texttt{not} \\
        &\qquad \mid \texttt{car} \mid \texttt{cdr} \mid \texttt{cons} \mid \texttt{nil} \mid \texttt{eq?} \mid \texttt{atom?} \mid \texttt{int?} \\
        &\qquad \mid \texttt{quote} \mid \texttt{quasiquote} \mid \texttt{unquote} \mid \texttt{begin} \\
    \langle \mathbf{identifier} \rangle &::= \langle \mathbf{word} \rangle
\end{align*}
$$ where the production for $\langle \mathbf{identifier} \rangle$ is restricted to those $\langle \mathbf{word} \rangle$ which are not a $\langle \mathbf{keyword} \rangle$.

The reader grammar is: $$
\begin{align*}
    \langle \mathbf{program\text{-}reader} \rangle &::= \langle \mathbf{datum\text{-}seq} \rangle \\
    \langle \mathbf{datum} \rangle &::= \langle \mathbf{atom} \rangle
        \mid \texttt{(} \langle \mathbf{datum\text{-}seq} \rangle \texttt{)}
        \mid \texttt{'} \langle \mathbf{datum} \rangle \\
        &\qquad \mid \texttt{`} \langle \mathbf{datum} \rangle
        \mid \texttt{,} \langle \mathbf{datum} \rangle \\
    \langle \mathbf{datum\text{-}seq} \rangle &::= \varepsilon \mid \langle \mathbf{datum} \rangle \langle \mathbf{datum\text{-}seq} \rangle \\
    \langle \mathbf{atom} \rangle &::= \langle \mathbf{integer} \rangle \mid \langle \mathbf{boolean} \rangle \mid \langle \mathbf{word} \rangle \mid \langle \mathbf{operator} \rangle \mid \texttt{.}
\end{align*}
$$ The three prefix forms are abbreviations: the reader takes $\texttt{'} \langle \mathbf{datum} \rangle$ to be $\texttt{(quote}\ \langle \mathbf{datum} \rangle \texttt{)}$, $\texttt{`} \langle \mathbf{datum} \rangle$ to be $\texttt{(quasiquote}\ \langle \mathbf{datum} \rangle \texttt{)}$, and $\texttt{,} \langle \mathbf{datum} \rangle$ to be $\texttt{(unquote}\ \langle \mathbf{datum} \rangle \texttt{)}$.

The evaluated subset is: $$
\begin{align*}
    \langle \mathbf{program\text{-}parser} \rangle &::= \langle \mathbf{form\text{-}seq} \rangle \\
    \langle \mathbf{form\text{-}seq} \rangle &::= \varepsilon \mid \langle \mathbf{form} \rangle \langle \mathbf{form\text{-}seq} \rangle \\
    \langle \mathbf{form} \rangle &::= \langle \mathbf{definition} \rangle \mid \langle \mathbf{expression} \rangle \\
    \langle \mathbf{definition} \rangle &::= \texttt{(} \texttt{define}\ \langle \mathbf{identifier} \rangle\ \langle \mathbf{expression} \rangle \texttt{)} \\
        &\qquad \mid \texttt{(} \texttt{define}\ \texttt{(} \langle \mathbf{identifier} \rangle\ \langle \mathbf{formals} \rangle \texttt{)}\ \langle \mathbf{expression} \rangle \texttt{)} \\
        &\qquad \mid \texttt{(} \texttt{begin}\ \langle \mathbf{definition\text{-}seq} \rangle \texttt{)} \\
    \langle \mathbf{definition\text{-}seq} \rangle &::= \varepsilon \mid \langle \mathbf{definition} \rangle \langle \mathbf{definition\text{-}seq} \rangle \\
    \langle \mathbf{expression} \rangle &::= \langle \mathbf{literal} \rangle
        \mid \langle \mathbf{identifier} \rangle
        \mid \langle \mathbf{quotation} \rangle
        \mid \langle \mathbf{quasiquotation} \rangle \\
        &\qquad \mid \langle \mathbf{lambda} \rangle
        \mid \langle \mathbf{conditional} \rangle
        \mid \langle \mathbf{sequence} \rangle \\
        &\qquad \mid \langle \mathbf{primitive\text{-}call} \rangle
        \mid \langle \mathbf{application} \rangle \\
    \langle \mathbf{quotation} \rangle &::= \texttt{(} \texttt{quote}\ \langle \mathbf{datum} \rangle \texttt{)} \mid \texttt{'} \langle \mathbf{datum} \rangle \\
    \langle \mathbf{quasiquotation} \rangle &::= \texttt{(} \texttt{quasiquote}\ \langle \mathbf{template} \rangle \texttt{)} \mid \texttt{`} \langle \mathbf{template} \rangle \\
    \langle \mathbf{sequence} \rangle &::= \texttt{(} \texttt{begin}\ \langle \mathbf{body} \rangle \texttt{)} \\
    \langle \mathbf{body} \rangle &::= \langle \mathbf{expression} \rangle \mid \langle \mathbf{form} \rangle \langle \mathbf{body} \rangle \\
    \langle \mathbf{literal} \rangle &::= \langle \mathbf{integer} \rangle \mid \langle \mathbf{boolean} \rangle \mid \texttt{nil} \\
    \langle \mathbf{lambda} \rangle &::= \texttt{(} \texttt{lambda}\ \texttt{(} \langle \mathbf{formals} \rangle \texttt{)}\ \langle \mathbf{expression} \rangle \texttt{)} \\
    \langle \mathbf{formals} \rangle &::= \langle \mathbf{identifier} \rangle \mid \langle \mathbf{identifier} \rangle \langle \mathbf{formals} \rangle \\
    \langle \mathbf{conditional} \rangle &::= \texttt{(} \texttt{if}\ \langle \mathbf{expression} \rangle\ \langle \mathbf{expression} \rangle\ \langle \mathbf{expression} \rangle \texttt{)} \\
    \langle \mathbf{primitive\text{-}call} \rangle &::= \texttt{(} \langle \mathbf{unary\text{-}prim} \rangle\ \langle \mathbf{expression} \rangle \texttt{)} \\
        &\qquad \mid \texttt{(} \langle \mathbf{binary\text{-}prim} \rangle\ \langle \mathbf{expression} \rangle\ \langle \mathbf{expression} \rangle \texttt{)} \\
    \langle \mathbf{unary\text{-}prim} \rangle &::= \texttt{car} \mid \texttt{cdr} \mid \texttt{atom?} \mid \texttt{int?} \\
    \langle \mathbf{binary\text{-}prim} \rangle &::= \texttt{cons} \mid \texttt{eq?} \mid \texttt{and} \mid \texttt{+} \mid \texttt{-} \mid \texttt{*} \mid \texttt{/} \\
    \langle \mathbf{application} \rangle &::= \texttt{(} \langle \mathbf{expression} \rangle\ \langle \mathbf{operands} \rangle \texttt{)} \\
    \langle \mathbf{operands} \rangle &::= \varepsilon \mid \langle \mathbf{expression} \rangle \langle \mathbf{operands} \rangle
\end{align*}
$$

And, finally, the grammar of quasiquotation templates is: $$
\begin{align*}
    \langle \mathbf{template} \rangle &::= \langle \mathbf{template\text{-}atom} \rangle
        \mid \texttt{(} \langle \mathbf{template\text{-}seq} \rangle \texttt{)}
        \mid \texttt{'} \langle \mathbf{template} \rangle \\
        &\qquad \mid \texttt{(} \texttt{quasiquote}\ \langle \mathbf{datum} \rangle \texttt{)}
        \mid \texttt{`} \langle \mathbf{datum} \rangle \\
        &\qquad \mid \texttt{(} \texttt{unquote}\ \langle \mathbf{expression} \rangle \texttt{)}
        \mid \texttt{,} \langle \mathbf{expression} \rangle \\
    \langle \mathbf{template\text{-}seq} \rangle &::= \varepsilon \mid \langle \mathbf{template} \rangle \langle \mathbf{template\text{-}seq} \rangle \\
    \langle \mathbf{template\text{-}atom} \rangle &::= \langle \mathbf{integer} \rangle \mid \langle \mathbf{boolean} \rangle \mid \langle \mathbf{operator} \rangle \mid \texttt{.} \mid \langle \mathbf{word} \rangle
\end{align*}
$$ where the $\langle \mathbf{word} \rangle$ in the last production is restricted to those which are neither $\texttt{quasiquote}$ nor $\texttt{unquote}$.

The reader and program parser can be operationalized as follows:

``` python
from dataclasses import dataclass

from lark import Lark, Token, Transformer, v_args
from lark.exceptions import (
    UnexpectedCharacters,
    UnexpectedEOF,
    UnexpectedInput,
    UnexpectedToken,
)
from lark.lexer import PatternStr

grammar = """
// plate grammar: one lexer shared by two start rules.
//   reader  -> plain datums (s-expressions)
//   program -> the evaluated subset
//
// Keyword terminals are named so the reader can keep them (a reference by
// name stays in the tree); the evaluated rules write them as string literals,
// which lark filters out of the tree.

// ================= Lexical grammar =================

DEFINE: "define"
LAMBDA: "lambda"
IF: "if"
LET: "let"
AND: "and"
OR: "or"
NOT: "not"
CAR: "car"
CDR: "cdr"
CONS: "cons"
NIL: "nil"
EQ: "eq?"
ATOM: "atom?"
INTP: "int?"
QUOTE: "quote"
QUASIQUOTE: "quasiquote"
UNQUOTE: "unquote"
BEGIN: "begin"

OPERATOR: "+" | "-" | "*" | "/"
BOOLEAN: "#t" | "#f"
INTEGER: /[0-9]+/
WORD: /[a-z][a-z?]*/
QUOTE_MARK: "'"
BACKQUOTE: "`"
COMMA: ","
DOT: "."

%import common.WS
%ignore WS

// ================= Reader grammar =================

reader: datum*

?datum: atom
      | "(" datum* ")"  -> list
      | "'" datum       -> quoted       // 'd reads as (quote d)
      | "`" datum       -> quasiquoted  // `d reads as (quasiquote d)
      | "," datum       -> unquoted     // ,d reads as (unquote d)

atom: INTEGER | BOOLEAN | WORD | _keyword | OPERATOR | DOT

_keyword: _template_keyword | QUASIQUOTE | UNQUOTE

_template_keyword: DEFINE | LAMBDA | IF | LET | AND | OR | NOT | CAR | CDR
                 | CONS | NIL | EQ | ATOM | INTP | QUOTE | BEGIN

// ================= Evaluated subset =================

program: form*

?form: definition | expression

?definition: "(" "define" WORD expression ")"                  -> define
           | "(" "define" "(" WORD formals ")" expression ")"  -> define_function
           | "(" "begin" definition* ")"                       -> define_block

?expression: INTEGER                                           -> integer
           | BOOLEAN                                           -> boolean
           | "nil"                                             -> nil
           | WORD                                              -> variable
           | "(" "quote" datum ")"                             -> quote
           | "'" datum                                         -> quote
           | "(" "quasiquote" template ")"                     -> quasiquote
           | "`" template                                      -> quasiquote
           | "(" "lambda" "(" formals ")" expression ")"       -> lambda_
           | "(" "if" expression expression expression ")"     -> conditional
           | "(" "begin" form* expression ")"                  -> sequence
           | "(" _unary_prim expression ")"                    -> primitive_call
           | "(" _binary_prim expression expression ")"        -> primitive_call
           | "(" expression expression* ")"                    -> application

formals: WORD+

_unary_prim: CAR | CDR | ATOM | INTP
_binary_prim: CONS | EQ | AND | OPERATOR

// A quasiquote template is data, except that `,e` / `(unquote e)` marks an
// expression to evaluate. `unquote` and `quasiquote` can't be plain atoms here,
// so `(unquote e)` has only one reading. A nested quasiquote is plain data.
?template: template_atom
         | "(" template* ")"                                   -> template_list
         | "'" template                                        -> template_quoted
         | "(" "quasiquote" datum ")"                          -> template_quasiquoted
         | "`" datum                                           -> template_quasiquoted
         | "(" "unquote" expression ")"                        -> unquote
         | "," expression                                      -> unquote

template_atom: INTEGER | BOOLEAN | WORD | _template_keyword | OPERATOR | DOT

"""

# ================= Reader datums =================


class Symbol(str):
    """A word, keyword, operator, `'` or `.` read as data."""

    __slots__ = ()

    def __repr__(self) -> str:
        return f"Symbol({str.__repr__(self)})"


# Lists are tuples. Note that bool is a subclass of int: test for bool first.
type Datum = int | bool | Symbol | tuple[Datum, ...]


# ================= Expressions =================


@dataclass(frozen=True)
class Integer:
    value: int


@dataclass(frozen=True)
class Boolean:
    value: bool


@dataclass(frozen=True)
class Nil:
    pass


@dataclass(frozen=True)
class Variable:
    name: str


@dataclass(frozen=True)
class Quote:
    """`(quote d)` or `'d`: the datum `d`, unevaluated."""

    datum: Datum


@dataclass(frozen=True)
class Unquote:
    """`,e` or `(unquote e)` inside a quasiquote template: `e` is evaluated."""

    expr: Expr


# A datum that may contain Unquote parts.
type Template = int | bool | Symbol | Unquote | tuple[Template, ...]


@dataclass(frozen=True)
class Quasiquote:
    """`` `t `` or `(quasiquote t)`: `t` is data except for its Unquote parts."""

    template: Template


@dataclass(frozen=True)
class Lambda:
    params: tuple[str, ...]
    body: Expr


@dataclass(frozen=True)
class If:
    test: Expr
    then: Expr
    orelse: Expr


@dataclass(frozen=True)
class Sequence:
    """`(begin ...)` in expression position; the last form is an expression."""

    forms: tuple[Form, ...]


@dataclass(frozen=True)
class PrimitiveCall:
    op: str
    args: tuple[Expr, ...]


@dataclass(frozen=True)
class Application:
    func: Expr
    args: tuple[Expr, ...]


type Expr = (
    Integer
    | Boolean
    | Nil
    | Variable
    | Quote
    | Quasiquote
    | Lambda
    | If
    | Sequence
    | PrimitiveCall
    | Application
)


# ================= Definitions =================


@dataclass(frozen=True)
class Define:
    """`(define name value)`; `(define (f x) e)` is stored as a Lambda value."""

    name: str
    value: Expr


@dataclass(frozen=True)
class DefineBlock:
    """`(begin <definition>...)`, which contains only definitions."""

    definitions: tuple[Definition, ...]


type Definition = Define | DefineBlock
type Form = Definition | Expr


@dataclass(frozen=True)
class Program:
    forms: tuple[Form, ...]

_LARK = Lark(
    grammar,
    start=["reader", "program"],
    parser="earley",
    lexer="basic",
)


class ParseError(Exception):
    """A syntax error, with the source position and the offending line."""

    def __init__(
        self,
        message: str,
        line: int,
        column: int,
        context: str,
        filename: str | None = None,
    ) -> None:
        super().__init__(message)
        self.message = message
        self.line = line
        self.column = column
        self.context = context
        self.filename = filename

    def __str__(self) -> str:
        # Source files get a compiler-style location; the REPL only needs the caret.
        if self.filename is None:
            header = self.message
        else:
            header = f"{self.filename}:{self.line}:{self.column}: {self.message}"
        return f"{header}\n{self.context}"


def read(source: str, filename: str | None = None) -> tuple[Datum, ...]:
    """Parse `source` with the reader grammar into a tuple of datums."""
    tree = _parse(source, "reader", filename)
    return _ReaderTransformer().transform(tree)


def parse(source: str, filename: str | None = None) -> Program:
    """Parse `source` as a program in the evaluated subset."""
    tree = _parse(source, "program", filename)
    return _SyntaxTransformer().transform(tree)


def _parse(source: str, start: str, filename: str | None):
    try:
        return _LARK.parse(source.lower(), start=start)
    except UnexpectedInput as e:
        raise _parse_error(e, source, filename) from None


# ================= Error reporting =================

_KEYWORDS = {
    "DEFINE", "LAMBDA", "IF", "LET", "AND", "OR", "NOT", "CAR",
    "CDR", "CONS", "NIL", "EQ", "ATOM", "INTP", "QUOTE", "QUASIQUOTE",
    "UNQUOTE", "BEGIN",
}  # fmt: skip

_PREFIXES = {"QUOTE_MARK", "BACKQUOTE", "COMMA"}
_DATUM_STARTERS = {"LPAR", "INTEGER", "BOOLEAN", "WORD", "OPERATOR", "DOT"} | _PREFIXES

# Terminals that can begin a datum / an expression. When all of them are
# expected, the error says "a datum" / "an expression" instead of listing each.
# Datums come first: they're a superset, and appear in both grammars (quote).
# Quasiquote templates are datums whose atoms can't be quasiquote/unquote.
_STARTERS = (
    ("a datum", _DATUM_STARTERS | _KEYWORDS),
    ("a datum", _DATUM_STARTERS | _KEYWORDS - {"QUASIQUOTE", "UNQUOTE"}),
    (
        "an expression",
        {"LPAR", "INTEGER", "BOOLEAN", "WORD", "NIL", "QUOTE_MARK", "BACKQUOTE"},
    ),
)

_TERMINAL_NAMES = {
    "INTEGER": "an integer",
    "BOOLEAN": "a boolean",
    "WORD": "an identifier",
    "OPERATOR": "an operator",
    "$END": "end of input",
}


def _parse_error(e: UnexpectedInput, source: str, filename: str | None) -> ParseError:
    match e:
        case UnexpectedCharacters():
            line, column = e.line, e.column
            message = f"unexpected character {source[e.pos_in_stream]!r}"
        case UnexpectedToken() if e.token.type != "$END":
            line, column = e.line, e.column
            text = source[e.token.start_pos : e.token.end_pos]
            message = f"unexpected {text!r}, expected {_describe(e.expected)}"
        case UnexpectedToken() | UnexpectedEOF():
            # Point just past the last non-blank character.
            before = source.rstrip()
            line = before.count("\n") + 1
            column = len(before) - (before.rfind("\n") + 1) + 1
            message = f"unexpected end of input, expected {_describe(e.expected)}"
        case _:
            line, column = e.line, e.column
            message = str(e)
    return ParseError(message, line, column, _context(source, line, column), filename)


def _describe(expected: set[str] | list[str]) -> str:
    names = set(expected)
    parts = []
    for noun, starters in _STARTERS:
        if starters <= names:
            names -= starters
            parts.append(noun)
    parts += sorted(_terminal_name(name) for name in names)
    if len(parts) == 1:
        return parts[0]
    return f"{', '.join(parts[:-1])} or {parts[-1]}"


def _terminal_name(name: str) -> str:
    if name in _TERMINAL_NAMES:
        return _TERMINAL_NAMES[name]
    pattern = _LARK.get_terminal(name).pattern
    return repr(pattern.value) if isinstance(pattern, PatternStr) else name


def _context(source: str, line: int, column: int) -> str:
    lines = source.splitlines() or [""]
    text = lines[min(line, len(lines)) - 1]
    # Keep tabs so the caret lines up with tab-indented source.
    pad = "".join(c if c == "\t" else " " for c in text[: column - 1])
    return f"    {text}\n    {pad}^"


# ================= Tree -> data =================


@v_args(inline=True)
class _ReaderTransformer(Transformer):
    def reader(self, *datums: Datum) -> tuple[Datum, ...]:
        return datums

    def list(self, *datums: Datum) -> tuple[Datum, ...]:
        return datums

    def quoted(self, datum: Datum) -> tuple[Datum, ...]:
        return (Symbol("quote"), datum)

    def quasiquoted(self, datum: Datum) -> tuple[Datum, ...]:
        return (Symbol("quasiquote"), datum)

    def unquoted(self, datum: Datum) -> tuple[Datum, ...]:
        return (Symbol("unquote"), datum)

    def atom(self, token: Token) -> Datum:
        match token.type:
            case "INTEGER":
                return int(token)
            case "BOOLEAN":
                return token == "#t"
            case _:
                return Symbol(token)


# Inherits the datum rules, which appear inside quote.
@v_args(inline=True)
class _SyntaxTransformer(_ReaderTransformer):
    def program(self, *forms):
        return Program(forms)

    def define(self, name: Token, value):
        return Define(str(name), value)

    def define_function(self, name: Token, params: tuple[str, ...], body):
        return Define(str(name), Lambda(params, body))

    def define_block(self, *definitions):
        return DefineBlock(definitions)

    def integer(self, token: Token):
        return Integer(int(token))

    def boolean(self, token: Token):
        return Boolean(token == "#t")

    def nil(self):
        return Nil()

    def variable(self, token: Token):
        return Variable(str(token))

    def quote(self, datum: Datum):
        return Quote(datum)

    def quasiquote(self, template: Template):
        return Quasiquote(template)

    template_atom = _ReaderTransformer.atom

    def template_list(self, *templates: Template) -> tuple[Template, ...]:
        return templates

    def template_quoted(self, template: Template) -> tuple[Template, ...]:
        return (Symbol("quote"), template)

    def template_quasiquoted(self, datum: Datum) -> tuple[Datum, ...]:
        return (Symbol("quasiquote"), datum)

    def unquote(self, expr):
        return Unquote(expr)

    def lambda_(self, params: tuple[str, ...], body):
        return Lambda(params, body)

    def conditional(self, test, then, orelse):
        return If(test, then, orelse)

    def sequence(self, *forms):
        return Sequence(forms)

    def primitive_call(self, op: Token, *args):
        return PrimitiveCall(str(op), args)

    def application(self, func, *args):
        return Application(func, args)

    def formals(self, *names: Token) -> tuple[str, ...]:
        return tuple(str(name) for name in names)
```

To demonstrate that the grammar is correct, we can produce an example:

In [7]:
from pprint import pprint

pprint(parse("(define (foo bar) bar)"))
pprint(parse("(+ 1 2)"))

Program(forms=(Define(name='foo',
                      value=Lambda(params=('bar',), body=Variable(name='bar'))),))
Program(forms=(PrimitiveCall(op='+',
                             args=(Integer(value=1), Integer(value=2))),))

## Encoding

### Basics of Language Encoding

-   Explain how we can encode $\mathcal{L}_\text{fruit}$

### Cleanup Memory

-   Cleanup memory and VSA wrapper

### Associative Memory

-   Associative memory and VSA wrapper

### Lists and Abstract Syntax

-   Talk about `VSAList`

### Lambdas

### Numbers

#### Peano Integer

#### aTRHC Integers

#### Quoting and quasiquotation

## Interpretation

### Semantics

### The Semantics of LISP$_\text{Plate}$

```` markdown
---
title: "The Plate Language"
authors:
  - name: Connor Hanley
    affiliation: Indiana University
    roles: writing
    corresponding: true
bibliography: references.bib
---

## Introduction

Thought seems to be structured. How can networks of connected units like
neurons have structure? McCulloch and Pitts demonstrated that, under certain assumptions, artificial 
neural networks are equivalent to a simple propositional logic [@mccullochPitts1942].
So too has it been demonstrated that neural networks can exhibit the same
systematic, compositional, and generative relations that psychological
states are taken to have [see @fodorConnectionismCognitiveArchitecture1988; 
@smolenskyTensorProductVariable1990; and also 
@plateHolographicReducedRepresentations1995; 
@gaylerVectorSymbolicArchitectures2004]. Likewise, programming
languages like the linearly typed lambda calculus [@velez-ginorioCompilingLinearNeurons2026;
and @velez-ginorioCompilingRecurrentNeurons2026] 

In this work we will describe an interpreter for the LISP programming language
which constructs and runs an equivalent network based on Vector Symbolic
Architectures (VSAs) [@tomkins-flanaganHeyPenttiWe2025; @hanleyHeyPenttiWe2025].
VSAs are a model of computation using high-dimensional vectors that support
compositional, systematic, and generative patterns 
[@smolenskyTensorProductVariable1990; @plateHolographicReducedRepresentations1995; 
@kanervaHyperdimensionalComputingIntroduction2009; @gaylerVectorSymbolicArchitectures2004].
We will demonstrate how VSAs are able to encode to (1) a subset of the LISP
language, called LISP$_\text{Plate}$ and (2) be ordered in a way so as to 
reproduce the semantics of a LISP$_\text{Plate}$.

To reach this point, first we will discuss VSAs in @sec-vsa, in particular, Holographic
Reduced Representations [HRRS, @plateHolographicReducedRepresentations1995]
and Time-domain Residue Hyperdimensional Computing [TRHC, @kymnComputingResidueNumbers2024].
Afterwards, we will describe a subset of the LISP language in @sec-lang, which
we will provide a mapping for to VSAs in @sec-enc. Finally, we will describe
how the mapped VSAs can be procedurally manipulated to reproduce the semantics
of LISP in @sec-interp.

## Vector Symbolic Architectures {#sec-vsa}

VSA/HDC is a implementation of computation in high-dimensional vectors
[or tensors, see @smolenskyTensorProductVariable1990] that provide neural
networks with the ability to construct and manipulate *distributed* patterns
that are compositional, systematic, and generative. Will provide a general
definition of VSAs, then focus in on the two VSAs that we will use to explore
list encodings: Holographic Reduced Representations [HRRs, from @plateHolographicReducedRepresentations1995]
and Time-domain Residue Hyperdimensional Computing [TRHC, counterpart to RHC from 
@kymnComputingResidueNumbers2024; cf. @voelkerSimulatingPredictingDynamical2021].

::: {#def-vsa}
A *Vector Symbolic Architecture* $\mathcal{V}$ is a structure $\langle X^{D_1 \times \dots \times D_n}, \sim,  \otimes, \oplus \rangle$,
where:

1. A *pattern set* $X^{D_1 \times \dots \times D_n}$,
2. There is a *similarity* function
$$
x_1 \sim x_2 : X^{D_1 \times \dots \times D_n} \times X^{D_1 \times \dots \times D_n} \to [0, 1];
$$
2. There is a *binding* function,
$$
x_1 \otimes x_2 : X^{D_1 \times \dots \times D_n} \times X^{D_1 \times \dots \times D_n} \to X^{D_1 \times \dots \times D_n},
$$
such that $x_1 \otimes x_2$ is neither similar to $x_1$ nor $x_2$, and the binding
operation admits an approximate inverse,
$$
\begin{align*}
    [x_1 \otimes^{-1} (x_1 \otimes x_2) \sim x_2] &\approx 1, \\
    [x_2 \otimes^{-1} (x_1 \otimes x_2) \sim x_1] &\approx 1;
\end{align*}
$$
3. And, there is a *bundling* or *superposition* operation,
$$
x_1 \oplus x_2 : X^{D_1 \times \dots \times D_n} \times X^{D_1 \times \dots \times D_n} \to X^{D_1 \times \dots \times D_n},
$$
such that $x_1 \oplus x_2$ is similar to both $x_1$ and $x_2$.
:::

Some VSAs include an extra *permutation* operation that functions like
a one-place binding, such that
$$
\Pi(x) : X^{D_1 \times \dots \times D_n} \to X^{D_1 \times \dots \times D_n},
$$
and,
$$
\Pi(x) \sim x \approx 0, \quad \Pi^{-1} \Pi (x) \sim x \approx 1.
$$
We will not include it in our definition, but we will use the permutation 
operation in the following.

@def-vsa can be operationalized by the following abstract class definition:
quarto-executable-code-5450563D

```python
#| label: lst-vsa-datatype
#| echo: true
from abc import ABCMeta, abstractmethod
from typing import ClassVar, Self

import numpy as np
import numpy.typing as npt

class VSA[T: np.generic](metaclass=ABCMeta):
    """Abstract base class of all VSA implementations.

    All VSAs must implement the following methods:
    """

    data: npt.NDArray[T]
    dtype: ClassVar[type[np.generic]]

    @staticmethod
    @abstractmethod
    def bind(x: npt.NDArray[T], y: npt.NDArray[T]) -> npt.NDArray[T]:
        """Vector symbolic binding."""
        ...

    @staticmethod
    @abstractmethod
    def bundle(x: npt.NDArray[T], y: npt.NDArray[T]) -> npt.NDArray[T]:
        """Vector symbolic bundling."""
        ...

    @staticmethod
    @abstractmethod
    def unbind(x: npt.NDArray[T], y: npt.NDArray[T]) -> npt.NDArray[T]:
        """Vector symbolic unbinding."""
        ...

    @staticmethod
    @abstractmethod
    def similarity(x: npt.NDArray[T], y: npt.NDArray[T]) -> float:
        """Vector symbolic similarity."""
        ...

    @classmethod
    @abstractmethod
    def new(cls, dim: int) -> Self:
        """Initialize a new vector."""
        ...

    @classmethod
    @abstractmethod
    def from_array(cls, array: npt.NDArray[T]) -> Self:
        """Create a VSA from an array."""
        ...

    @abstractmethod
    def __hash__(self) -> int: ...

```

### Holographic Reduced Representations

Holographic Reduced Representations [HRRs, @plateHolographicReducedRepresentations1995]
are a VSA defined over $D$-dimensional real vectors, which uses cosine similarity,
element-wise multiplication in the Fourier domain, and element-wise summation
to implement the VSA operations in @def-vsa. 

::: {#def-hrr}
## HRRs

*Holographic Reduced Representations* (HRRs) are a VSA $\mathcal{H}$, where

1. The set of high-dimensional patterns is $\mathbb{R}^D$;
2. The similarity function is *cosine similarity*:
$$
    x_1 \sim x_2 = \frac{x_1^\top x_2}{|x_1|_2 |x_2|_2};
$$
3. Binding is *circular convolution*:
$$
    x_1 \otimes x_2 = \mathcal{F}^{-1}\left[ \mathcal{F}(x_1) \odot \mathcal{F}(x_2)  \right],
$$
and unbinding *circular correlation*:
$$
x_1 \otimes^{-1} x_2 = \mathcal{F}^{-1} \left[ \mathcal{F}(x_1) \odot \overline{\mathcal{F}(x_2)} \right];
$$
4. And, bundling is the element-wise sum of $x_1$ and $x_2$:
$$
    x_1 \oplus x_2 = x_1 + x_2.
$$
:::

Any vector in the pattern set $\mathbb{R}^D$ of @def-hrr is able to have
the operations of (2-4) performed on it; however, in order to make the use
of the HRR's principled, assume in the following that newly generated HRR vectors
are unitary[^10]
Demonstrating that HRRs are VSAs aside from stipulation is trivial, so we will
leave it out of our consideration here.

[^10]: This is to aid in circular correlation.

@def-hrr can be operationalized as:
quarto-executable-code-5450563D

```python
#| label: lst-hrr
#| echo: true

import math
from typing import Literal, Self, cast, override

import numpy as np
import numpy.typing as npt
from numpy.fft import fft, ifft, irfft, rfft


type ArrayF64 = npt.NDArray[np.float64]

type Scheme = Literal["unitary", "gaussian"]
SCHEMES: tuple[Scheme, ...] = ("unitary", "gaussian")
"""How a fresh vector-symbol is drawn. Unitary vectors have a flat magnitude
spectrum, which makes `HRR.inv` their exact inverse rather than an approximate
one; Gaussian vectors are the classical choice.
"""


class HRR(VSA[np.float64]):
    """Holographic reduced representation vectors.

    The vectors of HRR are sampled from a normal distribution. Implements
    binding through circular convolution.
    """

    data: ArrayF64
    dtype = np.float64

    def __init__(self, data: ArrayF64) -> None:
        self.data = data

    @classmethod
    def normal(cls, size: int, sd: float | None = None) -> Self:
        """Create a new HRR by sampling from the normal distribution.

        Args:
            size (int): The dimensionality of the new HRR vector-symbol.
            sd (float | None): Defaults to `None`, the standard deviation
            of the normal distribution.

        Returns:
            A new HRR vector-symbol.
        """
        if sd is None:
            sd = 1.0 / math.sqrt(size)
        data = np.random.normal(scale=sd, size=size)
        data /= np.linalg.norm(data)
        return cls(data)

    @classmethod
    def unitary(cls, size: int) -> Self:
        """Create a new HRR whose magnitude spectrum is flat.

        Every frequency has a magnitude of one, so binding neither amplifies
        nor attenuates any of them: `HRR.inv` is then the exact inverse of
        binding rather than an approximate one, and the norm survives any
        number of bindings.

        Args:
            size (int): The dimensionality of the new HRR vector-symbol.

        Returns:
            A new unitary HRR vector-symbol.
        """
        spectrum = rfft(np.random.normal(size=size))
        return cls(irfft(spectrum / np.abs(spectrum), n=size))

    @override
    @classmethod
    def from_array(cls, array: ArrayF64) -> Self:
        """Create a new HRR from an array.

        Args:
            x (npt.NDArray[np.float64]): A raw float array.

        Returns:
            A new HRR vector-symbol, drawn from `x`.
        """
        return cls(array)

    @override
    @classmethod
    def new(cls, dim: int, scheme: Scheme = "unitary") -> Self:
        """Create a new vector-symbol.

        Args:
            dim (int): The dimensionality of the new vector-symbol.
            scheme (Scheme): Defaults to `"unitary"`, how the vector-symbol is
                drawn. See `SCHEMES`.

        Returns:
            A new HRR vector-symbol.

        Raises:
        -   ValueError: If `scheme` is not one of `SCHEMES`.
        """
        # Checked at runtime because the scheme reaches here as a string from
        # the command line, where the type is not enforced.
        if scheme not in SCHEMES:
            raise ValueError(f"scheme must be one of {SCHEMES}, got {scheme!r}")

        return cls.unitary(dim) if scheme == "unitary" else cls.normal(dim)

    @override
    @staticmethod
    def bind(x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """The product operation in the HRR VSA.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.
            y (npt.NDArray[np.float64]): The right-hand side of the operation.

        Returns:
            The circular convolution of `x` and `y`. Here, it is implemented
            through the fast Fourier transform.
        """
        return cast(ArrayF64, ifft(fft(x) * fft(y)).real)

    @override
    @staticmethod
    def bundle(x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """The HRR VSA sum operation.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.
            y (npt.NDArray[np.float64]): The right-hand side of the operation.

        Returns:
            The element-wise sum of the left-hand side and the right-hand side.
        """
        return x + y

    @staticmethod
    def inv(x: ArrayF64) -> ArrayF64:
        """The approximate inverse for HRR.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.

        Returns:
            The approximate inverse of `x`.
        """

        return x[np.r_[0, x.size - 1 : 0 : -1]]

    @override
    @staticmethod
    def unbind(x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """The unbinding operation in the HRR VSA.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.
            y (npt.NDArray[np.float64]): The right-hand side of the operation.

        Returns:
            The binding of the left-hand side with the approximate inverse
            of the right hand side.
        """
        return HRR.bind(x, HRR.inv(y))

    @override
    @staticmethod
    def similarity(x: ArrayF64, y: ArrayF64) -> float:
        """Approximated kernel for HRR. Measures the 'distance' between
        the left-hand and right-hand side.

        Args:
            x (npt.NDArray[np.float64]): The left-hand side of the operation.
            y (npt.NDArray[np.float64]): The right-hand side of the operation.

        Returns:
            The 'distance' between the left-hand side and the right-hand
            side, a value between -1 and 1.
        """
        mag = float(np.linalg.norm(x) * np.linalg.norm(y))
        if mag == 0.0:
            return 0.0
        else:
            return float(np.dot(x, y) / mag)
```

quarto-executable-code-5450563D

```python
#| include: false
from abc import update_abstractmethods


def HRRadd__(self, rhs: HRR | float) -> Self:
    """See `HRR.bundle`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.bundle(self.data, rhs.data))
    else:
        return cls(self.data + rhs)

def HRRradd__(self, rhs: HRR | float) -> Self:
    """See `HRR.bundle`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.bundle(self.data, rhs.data))
    else:
        return cls(self.data + rhs)

def HRRsub__(self, rhs: HRR | float) -> Self:
    """Element-wise subtraction."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(self.data - rhs.data)
    else:
        return cls(self.data - rhs)

def HRRmul__(self, rhs: HRR | float) -> Self:
    """Scalar multiplication or `HRR.bind`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.bind(self.data, rhs.data))
    else:
        return cls(self.data * rhs)

def HRRrmul__(self, rhs: HRR | float) -> Self:
    """Scalar multiplication or `HRR.bind`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.bind(self.data, rhs.data))
    else:
        return cls(self.data * rhs)

def HRRtruediv__(self, rhs: HRR | float) -> Self:
    """Scalar division or `HRR.unbind`."""
    cls = type(self)
    if isinstance(rhs, HRR):
        return cls(cls.unbind(self.data, rhs.data))
    elif isinstance(rhs, int):
        return cls(self.data / rhs)
    else:
        return cls((self.data / rhs).astype(np.float64))

def HRRinvert__(self) -> Self:
    """See `HRR.inv`."""
    cls = type(self)
    return cls(cls.inv(self.data))

def HRRneg__(self) -> Self:
    """Element-wise negation."""
    return type(self)(-self.data)

def magnitude(self) -> float:
    """The magnitude of the raw vector."""
    return math.sqrt(self.data @ self.data) / self.data.size

def HRRmatmul__(self, other: HRR | ArrayF64) -> float | ArrayF64:
    """Matrix multiplication."""
    if isinstance(other, HRR):
        return self.data @ other.data
    else:
        if len(other.shape) == 2:
            return (self.data @ other).astype(np.float64)
        else:
            return self.data @ other

def sim(self, other: HRR | ArrayF64) -> float:
    """See `HRR.similarity`."""
    if isinstance(other, HRR):
        return HRR.similarity(self.data, other.data)
    else:
        return HRR.similarity(self.data, other)

@override
def HRRstr__(self) -> str:
    return f"{type(self).__name__}({self.data})"

@override
def HRRhash__(self) -> int:
    return hash(self.data.tobytes())


for _method in (
    HRRadd__,
    HRRradd__,
    HRRsub__,
    HRRmul__,
    HRRrmul__,
    HRRtruediv__,
    HRRinvert__,
    HRRneg__,
    magnitude,
    HRRmatmul__,
    sim,
    HRRstr__,
    HRRhash__,
):
    name = str(_method.__name__).replace("HRR", "__")
    print(name)
    setattr(HRR, name, _method)

# `VSA.__hash__` is abstract, so recompute HRR's abstract methods now that
# it has been implemented.
_ = update_abstractmethods(HRR)
```

### Time-domain Residue Hyperdimensional Computing

Time-domain Residue Hyperdimensional Computing (TRHC) is a modification
of the Residue Hyperdimensional Computing VSA [RHC, @kymnComputingResidueNumbers2024]
which uses a residue number encoding to represent natural numbers and simple
arithmetic in VSAs. Strictly speaking, according to @def-vsa TRHC (and RHC) is
not one single VSA but two distinct VSAs, since TRHC defines two binding operations
that perform addition and multiplication over their encoded residue number.
For our purposes, we will focus only on the variant with additive binding
since the multiplicative binding operation is tedious to deal with.

Unlike HRRs, TRHC requires some preliminary definitions. Namely, since 
TRHC was developed to represent numbers using a residue encoding, we have
to first define what that means.

TRHC vectors must have the following properties:

1. TRHC vectors must have a unit norm, such that $\|x\| = 1$; and,
2. TRHC vectors must be *unitary under binding*, meaning that binding (circular
convolution) and unbinding (circular correlation) must be exact inverses.

Suppose we have a set of odd moduli $m_1, m_2, \dots, m_k$.
For each modulus, let $b_i$ denote the *base vector* of modulus $m_i$ of
dimension $D$. A base vector is constructed by[^1]:

1. If $D$ is even, then:
    + $b_i[0] = 0$,
    + $b_i[1, \dots, D / 2 - 1]$ are sampled from $\{0, \dots, m_i - 1\}$
    + $b_i[D / 2] = 0$, which is the *Nyquist* bin; and,
    + $b_i[D/2 + 1, \dots, D]$ is the mirror and flipped sign of $b_i[1, \dots, D/2-1]$.
2. If $D$ is odd, then:
    + $b_i[0] = 0$,
    + $b_i[1, \dots, \lfloor D / 2\rfloor]$ are sampled from $\{0, \dots, m_i - 1\}$, and
    + $b_i[\lfloor D / 2\rfloor + 1, \dots, D]$ are mirror and flipped sign of $b_i[1, \dots, \lfloor D/2 \rfloor]$.

To get the residue effect, we multiply by $2 \pi / m_i$ and exponentiate, and then
apply the inverse Fourier transform to return to the time domain:
$$
b_i = \exp\left( \frac{2 \pi b_i}{m_i} \right)
$$
For all base vectors, let $b_i^n$ denote:
$$
b_i^n = \mathcal{F}^{-1} \left[ \mathcal{F}(b_i)^n \right].
$$
Then, the encoding of a natural number $n$ in TRHC with moduli $m_1, m_2, \dots, m_k$ is:
$$
v(n) = b_1^n \otimes b_2^n \otimes \dots \otimes b_k^n,
$$
where $\cdot \otimes \cdot$ denotes circular convolution. This gives us the 
residue result, such that each vector stores the residues $n \bmod m_1, \dots, n \bmod m_k$,
and by the Chinese Remainder Theorem, $n$ is uniquely determined modulo
$\prod_i m_i$. 

[^1]: We use $b_i[\dots]$ to denote indexing into the vector, following `python` style, so as to not confuse the index into the vector with the index matching the modulus.

::: {#def-trhc}
## Additive Time-domain Residue Hyperdimensional Computing

Additive *Time-domain Residue Hyperdimensional Computing*[^2] (aTRHC) is a VSA with:

1. A set of $D$-dimensional real vectors, $\mathbb{R}^D$;
2. Similarity is *cosine similarity*;
3. Binding via circular convolution, unbinding via circular correlation; and,
4. Bundling via element-wise addition.
:::

aTRHC is said to be *additive* because circular convolution obeys the following
property:
$$
v(n) \otimes v(m) = v\left(n + m \bmod \prod_i m_i\right).
$$

[^2]: *Multiplicative* Binding is described for the Fourier-domain in @kymnComputingResidueNumbers2024.

@def-trhc can be operationalized as:
quarto-executable-code-5450563D

```python
#| label: lst-trhc
#| echo: true

from typing import ClassVar, Self, override

import numpy as np
import numpy.typing as npt


__all__ = ["TRHC"]

type ArrayF64 = npt.NDArray[np.float64]


class TRHC(HRR):
    """Time-domain Residue Hyperdimensional Computing.

    So-called, because it deals with RHC in the time-domain, as opposed to the frequency
    domain.
    """

    data: ArrayF64
    moduli: ClassVar[list[int]] = [3, 5, 7, 11]
    basis: ClassVar[list[ArrayF64]] = []

    @override
    @classmethod
    def new(cls, dim: int, scheme: Scheme = "unitary") -> Self:
        """Create a new vector-symbol.

        Args:
            dim (int): The dimensionality of the new vector-symbol.
            scheme (Scheme): Ignored. A Gaussian vector is not unitary, and so
                would leave the residue cycle as soon as it was bound to
                itself. The argument is kept only to match `HRR.new`.

        Returns:
            A new unitary TRHC vector-symbol.
        """
        return cls.unitary(dim)

    @staticmethod
    def generate_base_vector(
        rng: np.random.Generator, modulus: int, dim: int
    ) -> ArrayF64:
        """Generates an RHC base vector in the time domain.

        Args:
        -   rng (np.random.Generator): The random number generator.
        -   modulus (int): The modulus of the base vector.
        -   dim (int): The dimension of the base vector.

        Returns:
            An TRHC base vector in the time domain.
        """

        # Only the non-negative frequencies are drawn; `irfft` mirrors them
        # into the conjugate-symmetric half, which keeps the result real.
        k_choices = np.zeros(dim // 2 + 1, dtype=int)
        k_choices[0] = 0
        k_choices[1:] = rng.choice(modulus, dim // 2)

        if dim % 2 == 0:
            # The Nyquist bin is its own conjugate, so its phase must be 0 or
            # pi. Odd moduli admit no phase of pi, leaving 0 as the only choice.
            k_choices[-1] = 0

        phases = 2 * np.pi * k_choices / modulus
        return np.fft.irfft(np.exp(1j * phases), n=dim)

    @classmethod
    def generate_basis(cls, dim: int) -> None:
        """Generate a fresh basis, one base vector per modulus.

        Args:
        -   dim (int): The dimension of the base vectors.
        """
        rng = np.random.default_rng()
        cls.basis = [cls.generate_base_vector(rng, mod, dim) for mod in cls.moduli]

    @staticmethod
    def _bind_power(vec: ArrayF64, num: int) -> ArrayF64:
        """Raise a base vector to an integer binding power.

        Equivalent to binding `vec` with itself `num` times, but evaluated in the
        frequency domain so the result stays unitary for any `num`. A power of 0
        gives the binding identity.

        Args:
        -   vec (ArrayF64): The base vector.
        -   num (int): The binding power.

        Returns:
            The base vector raised to the given binding power.
        """
        return np.fft.ifft(np.fft.fft(vec) ** num).real

    @classmethod
    def number(
        cls,
        num: int,
        dim: int,
        alternative_basis: list[ArrayF64] | None = None,
    ) -> Self:
        """Create an TRHC vector from a number.

        Args:
        -   num (int): The number to convert.
        -   dim (int): The dimension of the vector.
        -   alternative_basis (list[ArrayF64] | None): An optional alternative basis to use.

        Returns:
            An TRHC vector representing the number.

        Raises:
        -   ValueError: If `alternative_basis` is provided and is empty.
        -   ValueError: If the basis dimension does not match the dimension of the vector.
        """

        if not cls.basis:
            cls.generate_basis(dim)

        basis = cls.basis

        if alternative_basis is not None and len(alternative_basis) == 0:
            raise ValueError("alternative_basis must not be empty")
        elif alternative_basis is not None and isinstance(
            alternative_basis[0], np.ndarray
        ):
            basis = alternative_basis

        if basis[0].shape[0] != dim:
            raise ValueError("basis dimension must match dim")

        rhc_num = cls._bind_power(basis[0], num)
        for i in range(1, len(basis)):
            rhc_num = cls.bind(rhc_num, cls._bind_power(basis[i], num))

        return cls(rhc_num)

    @classmethod
    def residue_add(cls, x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """Perform TRHC arithmetical addition.

        Args:
        -   x (ArrayF64): The first vector.
        -   y (ArrayF64): The second vector.

        Returns:
            The result of the addition.
        """
        return cls.bind(x, y)

    @classmethod
    def residue_sub(cls, x: ArrayF64, y: ArrayF64) -> ArrayF64:
        """Perform TRHC arithmetical subtraction.

        Args:
        -   x (ArrayF64): The first vector.
        -   y (ArrayF64): The second vector.

        Returns:
            The result of the subtraction.
        """
        return cls.unbind(x, y)

```

aTRHC will be important later on for us when we deal with numerical
representations in @sec-numbers, where we will compare its abilities to
an alternative Peano encoding.

## The Language {#sec-lang}

VSAs exhibit compositional and systematic relations in virtue of operations
(2-4) in @def-vsa. They can also be used to encode data structures, such as lists,
graphs, and trees [@kleykoSurveyHyperdimensionalComputing2023]. Following
@tomkins-flanaganHeyPenttiWe2025 and @hanleyHeyPenttiWe2025, we will
instead encode and interpret a full-fledged programming language. In particular,
we will be encoding a subset of the R5RS Scheme standard [@kelseyRevisedReport1998].

### What is a Language

More formally speaking, a language $\mathcal{L}$ is a set of strings that are
closed under a recursive definition of set membership. We will call the
sentences that must hold true of a string the *formation rules* of $\mathcal{L}$.
Likewise, a string which obeys these formation rules will be called 
a *well-formed formula of* $\mathcal{L}$. 

::: {#def-language}
## Language

A *language* $\mathcal{L}$ is a set of strings defined by a set of 
*formation rules* $S_1, S_2, \dots, S_n$, such that strings are only in the
set iff it is true of them that $S_1, S_2, \dots, S_n$. A string that is in
the language $\mathcal{L}$ is said to be a *well-formed formula of* $\mathcal{L}$.
:::

To give a toy example, let us discuss the lanaguage $\mathcal{L}_\text{fruit}$.

::: {#def-lang-fruit}
Let $\mathcal{A}$ be the set of atomic symbols $\{\text{apple}, \text{banana}, \text{pear}\}$.
Then, the language $\mathcal{L}_\text{fruit}$ is given by the following formation
rules:

1. (Atomic formulae) If $a \in \mathcal{A}$, then $a \in \mathcal{L}_\text{fruit}$
2. (Compound formulae) If $x_1, x_2 \in \mathcal{L}_\text{fruit}$, then the
*disjunction* of $x_1$ and $x_2$, $(x_1 \lor x_2) \in \mathcal{L}_\text{fruit}$
and the *conjunction* of $x_1$ and $x_2$, $(x_2 \land x_2) \in \mathcal{L}_\text{fruit}$.
3. (Closure Under the Formation Rules) No other string is in $\mathcal{L}_\text{fruit}$.
:::

This definition gives us simple sentences like $\text{apple}$, but also 
complex sentences like $((\text{apple} \lor (\text{banana} \land \text{pear})) \land (\text{banana} \lor \text{apple}))$. Not a very useful language except for
maybe expressing personal preferences[^11], however the coupling of the language
definition with formation rules allows for us to express the language even
more tersely. Instead of providing an explicit definition of formation rules,
we can provide a *grammar* of a language $\mathcal{L}$ by writing it out
in Backus-Naur Form.

[^11]: Though, it should be clear that one could define really any relevant 
language in the provided manner. For example, propositional logic is not too
far off from our definition of $\mathcal{L}_\text{fruit}$.

::: {#def-bnf}

## Grammar; BNF

A *grammar* of a language $\mathcal{L}$ is a 
$4$-tuple $G = \langle N, \Sigma, P, S, \rangle$, where:

1. $N$ is a finite set of *non-terminals*, written $\langle \dots \rangle$;
2. $\Sigma$ isa a finite set of *terminals*, the atomic symbols of the language,
   where $N \cap \Sigma = \varnothing$;
3. $P$ is a finite set of *productions*, each with the form $A ::= \alpha$,
for $A \in N$ and $\alpha \in (N \cup \Sigma)^*$; and,
4. $S \in N$ is a *start symbol*.

We write $V = N \cup \Sigma$ and let $\varepsilon$ denote the empty string.
The *Backus-Naur Form* (BNF) is a notation for $P$ in which several productions
sharing a left-hand side are collapsed into a single rule by a bar: the rule
$A ::= \alpha_1 \mid \alpha_2 \mid \dots \mid \alpha_n$ abbreviate the $n$
productions $A ::= \alpha_i$.

$G$ *derives* $\gamma$ from $\beta$ in one step, denoted by $\beta \Rightarrow \gamma$,
when $\beta = \mu A \nu$ and $\gamma = \alpha$ in $P$; i.e., when
$\gamma$ is the result of replacing one nonterminal occurrence in $\beta$ by
the right-hand side of one of its rules. Let $\Rightarrow^*$ denote the
reflexive transitive closure of $\Rightarrow$. Then, the language generated by
$G$ is:
$$
\mathcal{L}(G) = \{w \in \Sigma^* : S \Rightarrow^* w\}.
$$
:::

Since $\mathcal{L}(G)$ in @def-bnf is defined for exactly the terminal strings
derivable from $S$, a grammar carries the clsoure property clause ((3) of @def-lang-fruit)
for free.

::: {#def-fruit-bnf}

## Grammar of $\mathcal{L}_\text{fruit}$

The grammar of $\mathcal{L}_\text{fruit}$ is given by:

1. The set of non-terminals $N$:
$$
N = \{ \langle \mathbf{atomic} \rangle, \langle \mathbf{disjunction} \rangle, \langle \mathbf{conjunction} \rangle, \langle \textbf{expression} \rangle  \}
$$

2. The set of terminal symbols $\Sigma$:
$$
\Sigma = \{ \text{apple}, \text{banana}, \text{pear}, \land, \lor \}
$$

3. The set of productions $P$:
$$
\begin{align*}
P =  \{&\langle \mathbf{atomic} \rangle ::= \text{apple} \mid \text{banana} \mid \text{pear}, \\
    &\langle \mathbf{conjunction} \rangle ::= \langle \textbf{expression} \rangle \land \langle \textbf{expression}, \rangle \\
    &\langle \mathbf{disjunction} \rangle ::= \langle \textbf{expression} \rangle \lor \langle \textbf{expression}, \rangle \\
    &\langle \mathbf{expression} \rangle ::= \langle \mathbf{atomic} \rangle \mid \langle \mathbf{disjunction} \rangle \mid \langle \mathbf{conjunction} \rangle
\}
\end{align*}
$$

4. The start symbol $S$ is $\langle \mathbf{expression} \rangle$.
:::

The BNF of @def-fruit-bnf can be sufficiently inferred merely from the set
of production rules:
$$
\begin{align*}
    \langle \mathbf{atomic} \rangle &::= \text{apple} \mid \text{banana} \mid \text{pear}, \\
    \langle \mathbf{conjunction} \rangle &::= \langle \textbf{expression} \rangle \land \langle \textbf{expression}, \rangle \\
    \langle \mathbf{disjunction} \rangle &::= \langle \textbf{expression} \rangle \lor \langle \textbf{expression}, \rangle \\
    \langle \mathbf{expression} \rangle &::= \langle \mathbf{atomic} \rangle \mid \langle \mathbf{disjunction} \rangle \mid \langle \mathbf{conjunction} \rangle
\end{align*}
$$

BNF grammars already conveniently give a way to easily express the grammar
of a language using algebraic data types, or tagged unions. Our grammar for 
$\mathcal{L}_\text{fruit}$ can be operationalized as follows:
quarto-executable-code-5450563D

```python
#| label: lst-fruit
#| echo: true
#| eval: false
#| code-fold: false

from dataclasses import dataclass
from typing import Literal

@dataclass
class Atom:
    x: Literal["apple", "banana", "pear"]


@dataclass
class Conjunction:
    lhs: "Expression"
    rhs: "Expression"


@dataclass
class Disjunction:
    lhs: "Expression"
    rhs: "Expression"


type Expression = Atom | Conjunction | Disjunction
```

Note, in @lst-fruit we ignore the conjunction and disjunction symbols. This
purely for the sake of convenience: the fact that we separate the non-terminal
productions gives us sufficient information about the form so as to distinguish
them. If we modified the production rules to have a single $\langle \mathbf{compound} \rangle$
production rule, then we would have to keep the disjunction and conjunction symbols
in our operationalization.

### The Grammar of LISP$_\text{Plate}$

Following @def-fruit-bnf, we give the grammar of LISP$_\text{Plate}$ by its
production rules alone. The rules fall into four groups: a *lexical* grammar
that fixes the tokens, a *reader* grammar that turns a sequence of tokens into
data, an *evaluated* subset that picks out the data which are also programs, and
a small grammar of quasiquotation templates. Both the reader and the evaluated
subset have a start symbol for whole programs; we distinguish them by writing
$\langle \mathbf{program\text{-}reader} \rangle$ and
$\langle \mathbf{program\text{-}parser} \rangle$ respectively.

The lexical grammar assumes that the input has been lower-cased and that
whitespace between tokens is discarded:
$$
\begin{align*}
    \langle \mathbf{token} \rangle &::= \texttt{(} \mid \texttt{)} \mid \texttt{'} \mid \texttt{`} \mid \texttt{,} \mid \texttt{.} \mid \langle \mathbf{operator} \rangle \\
        &\qquad \mid \langle \mathbf{boolean} \rangle \mid \langle \mathbf{integer} \rangle \mid \langle \mathbf{word} \rangle \\
    \langle \mathbf{operator} \rangle &::= \texttt{+} \mid \texttt{-} \mid \texttt{*} \mid \texttt{/} \\
    \langle \mathbf{boolean} \rangle &::= \texttt{\#t} \mid \texttt{\#f} \\
    \langle \mathbf{integer} \rangle &::= \langle \mathbf{digit} \rangle \mid \langle \mathbf{digit} \rangle \langle \mathbf{integer} \rangle \\
    \langle \mathbf{word} \rangle &::= \langle \mathbf{letter} \rangle \langle \mathbf{word\text{-}tail} \rangle \\
    \langle \mathbf{word\text{-}tail} \rangle &::= \varepsilon \mid \langle \mathbf{letter} \rangle \langle \mathbf{word\text{-}tail} \rangle \mid \texttt{?} \langle \mathbf{word\text{-}tail} \rangle \\
    \langle \mathbf{letter} \rangle &::= \texttt{a} \mid \texttt{b} \mid \dots \mid \texttt{z} \\
    \langle \mathbf{digit} \rangle &::= \texttt{0} \mid \texttt{1} \mid \dots \mid \texttt{9} \\
    \langle \mathbf{keyword} \rangle &::= \texttt{define} \mid \texttt{lambda} \mid \texttt{if} \mid \texttt{let} \mid \texttt{and} \mid \texttt{or} \mid \texttt{not} \\
        &\qquad \mid \texttt{car} \mid \texttt{cdr} \mid \texttt{cons} \mid \texttt{nil} \mid \texttt{eq?} \mid \texttt{atom?} \mid \texttt{int?} \\
        &\qquad \mid \texttt{quote} \mid \texttt{quasiquote} \mid \texttt{unquote} \mid \texttt{begin} \\
    \langle \mathbf{identifier} \rangle &::= \langle \mathbf{word} \rangle
\end{align*}
$$
where the production for $\langle \mathbf{identifier} \rangle$ is restricted to
those $\langle \mathbf{word} \rangle$ which are not a $\langle \mathbf{keyword} \rangle$.

The reader grammar is:
$$
\begin{align*}
    \langle \mathbf{program\text{-}reader} \rangle &::= \langle \mathbf{datum\text{-}seq} \rangle \\
    \langle \mathbf{datum} \rangle &::= \langle \mathbf{atom} \rangle
        \mid \texttt{(} \langle \mathbf{datum\text{-}seq} \rangle \texttt{)}
        \mid \texttt{'} \langle \mathbf{datum} \rangle \\
        &\qquad \mid \texttt{`} \langle \mathbf{datum} \rangle
        \mid \texttt{,} \langle \mathbf{datum} \rangle \\
    \langle \mathbf{datum\text{-}seq} \rangle &::= \varepsilon \mid \langle \mathbf{datum} \rangle \langle \mathbf{datum\text{-}seq} \rangle \\
    \langle \mathbf{atom} \rangle &::= \langle \mathbf{integer} \rangle \mid \langle \mathbf{boolean} \rangle \mid \langle \mathbf{word} \rangle \mid \langle \mathbf{operator} \rangle \mid \texttt{.}
\end{align*}
$$
The three prefix forms are abbreviations: the reader takes
$\texttt{'} \langle \mathbf{datum} \rangle$ to be
$\texttt{(quote}\ \langle \mathbf{datum} \rangle \texttt{)}$,
$\texttt{`} \langle \mathbf{datum} \rangle$ to be
$\texttt{(quasiquote}\ \langle \mathbf{datum} \rangle \texttt{)}$, and
$\texttt{,} \langle \mathbf{datum} \rangle$ to be
$\texttt{(unquote}\ \langle \mathbf{datum} \rangle \texttt{)}$.

The evaluated subset is:
$$
\begin{align*}
    \langle \mathbf{program\text{-}parser} \rangle &::= \langle \mathbf{form\text{-}seq} \rangle \\
    \langle \mathbf{form\text{-}seq} \rangle &::= \varepsilon \mid \langle \mathbf{form} \rangle \langle \mathbf{form\text{-}seq} \rangle \\
    \langle \mathbf{form} \rangle &::= \langle \mathbf{definition} \rangle \mid \langle \mathbf{expression} \rangle \\
    \langle \mathbf{definition} \rangle &::= \texttt{(} \texttt{define}\ \langle \mathbf{identifier} \rangle\ \langle \mathbf{expression} \rangle \texttt{)} \\
        &\qquad \mid \texttt{(} \texttt{define}\ \texttt{(} \langle \mathbf{identifier} \rangle\ \langle \mathbf{formals} \rangle \texttt{)}\ \langle \mathbf{expression} \rangle \texttt{)} \\
        &\qquad \mid \texttt{(} \texttt{begin}\ \langle \mathbf{definition\text{-}seq} \rangle \texttt{)} \\
    \langle \mathbf{definition\text{-}seq} \rangle &::= \varepsilon \mid \langle \mathbf{definition} \rangle \langle \mathbf{definition\text{-}seq} \rangle \\
    \langle \mathbf{expression} \rangle &::= \langle \mathbf{literal} \rangle
        \mid \langle \mathbf{identifier} \rangle
        \mid \langle \mathbf{quotation} \rangle
        \mid \langle \mathbf{quasiquotation} \rangle \\
        &\qquad \mid \langle \mathbf{lambda} \rangle
        \mid \langle \mathbf{conditional} \rangle
        \mid \langle \mathbf{sequence} \rangle \\
        &\qquad \mid \langle \mathbf{primitive\text{-}call} \rangle
        \mid \langle \mathbf{application} \rangle \\
    \langle \mathbf{quotation} \rangle &::= \texttt{(} \texttt{quote}\ \langle \mathbf{datum} \rangle \texttt{)} \mid \texttt{'} \langle \mathbf{datum} \rangle \\
    \langle \mathbf{quasiquotation} \rangle &::= \texttt{(} \texttt{quasiquote}\ \langle \mathbf{template} \rangle \texttt{)} \mid \texttt{`} \langle \mathbf{template} \rangle \\
    \langle \mathbf{sequence} \rangle &::= \texttt{(} \texttt{begin}\ \langle \mathbf{body} \rangle \texttt{)} \\
    \langle \mathbf{body} \rangle &::= \langle \mathbf{expression} \rangle \mid \langle \mathbf{form} \rangle \langle \mathbf{body} \rangle \\
    \langle \mathbf{literal} \rangle &::= \langle \mathbf{integer} \rangle \mid \langle \mathbf{boolean} \rangle \mid \texttt{nil} \\
    \langle \mathbf{lambda} \rangle &::= \texttt{(} \texttt{lambda}\ \texttt{(} \langle \mathbf{formals} \rangle \texttt{)}\ \langle \mathbf{expression} \rangle \texttt{)} \\
    \langle \mathbf{formals} \rangle &::= \langle \mathbf{identifier} \rangle \mid \langle \mathbf{identifier} \rangle \langle \mathbf{formals} \rangle \\
    \langle \mathbf{conditional} \rangle &::= \texttt{(} \texttt{if}\ \langle \mathbf{expression} \rangle\ \langle \mathbf{expression} \rangle\ \langle \mathbf{expression} \rangle \texttt{)} \\
    \langle \mathbf{primitive\text{-}call} \rangle &::= \texttt{(} \langle \mathbf{unary\text{-}prim} \rangle\ \langle \mathbf{expression} \rangle \texttt{)} \\
        &\qquad \mid \texttt{(} \langle \mathbf{binary\text{-}prim} \rangle\ \langle \mathbf{expression} \rangle\ \langle \mathbf{expression} \rangle \texttt{)} \\
    \langle \mathbf{unary\text{-}prim} \rangle &::= \texttt{car} \mid \texttt{cdr} \mid \texttt{atom?} \mid \texttt{int?} \\
    \langle \mathbf{binary\text{-}prim} \rangle &::= \texttt{cons} \mid \texttt{eq?} \mid \texttt{and} \mid \texttt{+} \mid \texttt{-} \mid \texttt{*} \mid \texttt{/} \\
    \langle \mathbf{application} \rangle &::= \texttt{(} \langle \mathbf{expression} \rangle\ \langle \mathbf{operands} \rangle \texttt{)} \\
    \langle \mathbf{operands} \rangle &::= \varepsilon \mid \langle \mathbf{expression} \rangle \langle \mathbf{operands} \rangle
\end{align*}
$$

And, finally, the grammar of quasiquotation templates is:
$$
\begin{align*}
    \langle \mathbf{template} \rangle &::= \langle \mathbf{template\text{-}atom} \rangle
        \mid \texttt{(} \langle \mathbf{template\text{-}seq} \rangle \texttt{)}
        \mid \texttt{'} \langle \mathbf{template} \rangle \\
        &\qquad \mid \texttt{(} \texttt{quasiquote}\ \langle \mathbf{datum} \rangle \texttt{)}
        \mid \texttt{`} \langle \mathbf{datum} \rangle \\
        &\qquad \mid \texttt{(} \texttt{unquote}\ \langle \mathbf{expression} \rangle \texttt{)}
        \mid \texttt{,} \langle \mathbf{expression} \rangle \\
    \langle \mathbf{template\text{-}seq} \rangle &::= \varepsilon \mid \langle \mathbf{template} \rangle \langle \mathbf{template\text{-}seq} \rangle \\
    \langle \mathbf{template\text{-}atom} \rangle &::= \langle \mathbf{integer} \rangle \mid \langle \mathbf{boolean} \rangle \mid \langle \mathbf{operator} \rangle \mid \texttt{.} \mid \langle \mathbf{word} \rangle
\end{align*}
$$
where the $\langle \mathbf{word} \rangle$ in the last production is restricted to
those which are neither $\texttt{quasiquote}$ nor $\texttt{unquote}$.

The reader and program parser can be operationalized as follows:
quarto-executable-code-5450563D

```python
#| label: lst-parsing
#| eval: true
#| echo: true

from dataclasses import dataclass

from lark import Lark, Token, Transformer, v_args
from lark.exceptions import (
    UnexpectedCharacters,
    UnexpectedEOF,
    UnexpectedInput,
    UnexpectedToken,
)
from lark.lexer import PatternStr

grammar = """
// plate grammar: one lexer shared by two start rules.
//   reader  -> plain datums (s-expressions)
//   program -> the evaluated subset
//
// Keyword terminals are named so the reader can keep them (a reference by
// name stays in the tree); the evaluated rules write them as string literals,
// which lark filters out of the tree.

// ================= Lexical grammar =================

DEFINE: "define"
LAMBDA: "lambda"
IF: "if"
LET: "let"
AND: "and"
OR: "or"
NOT: "not"
CAR: "car"
CDR: "cdr"
CONS: "cons"
NIL: "nil"
EQ: "eq?"
ATOM: "atom?"
INTP: "int?"
QUOTE: "quote"
QUASIQUOTE: "quasiquote"
UNQUOTE: "unquote"
BEGIN: "begin"

OPERATOR: "+" | "-" | "*" | "/"
BOOLEAN: "#t" | "#f"
INTEGER: /[0-9]+/
WORD: /[a-z][a-z?]*/
QUOTE_MARK: "'"
BACKQUOTE: "`"
COMMA: ","
DOT: "."

%import common.WS
%ignore WS

// ================= Reader grammar =================

reader: datum*

?datum: atom
      | "(" datum* ")"  -> list
      | "'" datum       -> quoted       // 'd reads as (quote d)
      | "`" datum       -> quasiquoted  // `d reads as (quasiquote d)
      | "," datum       -> unquoted     // ,d reads as (unquote d)

atom: INTEGER | BOOLEAN | WORD | _keyword | OPERATOR | DOT

_keyword: _template_keyword | QUASIQUOTE | UNQUOTE

_template_keyword: DEFINE | LAMBDA | IF | LET | AND | OR | NOT | CAR | CDR
                 | CONS | NIL | EQ | ATOM | INTP | QUOTE | BEGIN

// ================= Evaluated subset =================

program: form*

?form: definition | expression

?definition: "(" "define" WORD expression ")"                  -> define
           | "(" "define" "(" WORD formals ")" expression ")"  -> define_function
           | "(" "begin" definition* ")"                       -> define_block

?expression: INTEGER                                           -> integer
           | BOOLEAN                                           -> boolean
           | "nil"                                             -> nil
           | WORD                                              -> variable
           | "(" "quote" datum ")"                             -> quote
           | "'" datum                                         -> quote
           | "(" "quasiquote" template ")"                     -> quasiquote
           | "`" template                                      -> quasiquote
           | "(" "lambda" "(" formals ")" expression ")"       -> lambda_
           | "(" "if" expression expression expression ")"     -> conditional
           | "(" "begin" form* expression ")"                  -> sequence
           | "(" _unary_prim expression ")"                    -> primitive_call
           | "(" _binary_prim expression expression ")"        -> primitive_call
           | "(" expression expression* ")"                    -> application

formals: WORD+

_unary_prim: CAR | CDR | ATOM | INTP
_binary_prim: CONS | EQ | AND | OPERATOR

// A quasiquote template is data, except that `,e` / `(unquote e)` marks an
// expression to evaluate. `unquote` and `quasiquote` can't be plain atoms here,
// so `(unquote e)` has only one reading. A nested quasiquote is plain data.
?template: template_atom
         | "(" template* ")"                                   -> template_list
         | "'" template                                        -> template_quoted
         | "(" "quasiquote" datum ")"                          -> template_quasiquoted
         | "`" datum                                           -> template_quasiquoted
         | "(" "unquote" expression ")"                        -> unquote
         | "," expression                                      -> unquote

template_atom: INTEGER | BOOLEAN | WORD | _template_keyword | OPERATOR | DOT

"""

# ================= Reader datums =================


class Symbol(str):
    """A word, keyword, operator, `'` or `.` read as data."""

    __slots__ = ()

    def __repr__(self) -> str:
        return f"Symbol({str.__repr__(self)})"


# Lists are tuples. Note that bool is a subclass of int: test for bool first.
type Datum = int | bool | Symbol | tuple[Datum, ...]


# ================= Expressions =================


@dataclass(frozen=True)
class Integer:
    value: int


@dataclass(frozen=True)
class Boolean:
    value: bool


@dataclass(frozen=True)
class Nil:
    pass


@dataclass(frozen=True)
class Variable:
    name: str


@dataclass(frozen=True)
class Quote:
    """`(quote d)` or `'d`: the datum `d`, unevaluated."""

    datum: Datum


@dataclass(frozen=True)
class Unquote:
    """`,e` or `(unquote e)` inside a quasiquote template: `e` is evaluated."""

    expr: Expr


# A datum that may contain Unquote parts.
type Template = int | bool | Symbol | Unquote | tuple[Template, ...]


@dataclass(frozen=True)
class Quasiquote:
    """`` `t `` or `(quasiquote t)`: `t` is data except for its Unquote parts."""

    template: Template


@dataclass(frozen=True)
class Lambda:
    params: tuple[str, ...]
    body: Expr


@dataclass(frozen=True)
class If:
    test: Expr
    then: Expr
    orelse: Expr


@dataclass(frozen=True)
class Sequence:
    """`(begin ...)` in expression position; the last form is an expression."""

    forms: tuple[Form, ...]


@dataclass(frozen=True)
class PrimitiveCall:
    op: str
    args: tuple[Expr, ...]


@dataclass(frozen=True)
class Application:
    func: Expr
    args: tuple[Expr, ...]


type Expr = (
    Integer
    | Boolean
    | Nil
    | Variable
    | Quote
    | Quasiquote
    | Lambda
    | If
    | Sequence
    | PrimitiveCall
    | Application
)


# ================= Definitions =================


@dataclass(frozen=True)
class Define:
    """`(define name value)`; `(define (f x) e)` is stored as a Lambda value."""

    name: str
    value: Expr


@dataclass(frozen=True)
class DefineBlock:
    """`(begin <definition>...)`, which contains only definitions."""

    definitions: tuple[Definition, ...]


type Definition = Define | DefineBlock
type Form = Definition | Expr


@dataclass(frozen=True)
class Program:
    forms: tuple[Form, ...]

_LARK = Lark(
    grammar,
    start=["reader", "program"],
    parser="earley",
    lexer="basic",
)


class ParseError(Exception):
    """A syntax error, with the source position and the offending line."""

    def __init__(
        self,
        message: str,
        line: int,
        column: int,
        context: str,
        filename: str | None = None,
    ) -> None:
        super().__init__(message)
        self.message = message
        self.line = line
        self.column = column
        self.context = context
        self.filename = filename

    def __str__(self) -> str:
        # Source files get a compiler-style location; the REPL only needs the caret.
        if self.filename is None:
            header = self.message
        else:
            header = f"{self.filename}:{self.line}:{self.column}: {self.message}"
        return f"{header}\n{self.context}"


def read(source: str, filename: str | None = None) -> tuple[Datum, ...]:
    """Parse `source` with the reader grammar into a tuple of datums."""
    tree = _parse(source, "reader", filename)
    return _ReaderTransformer().transform(tree)


def parse(source: str, filename: str | None = None) -> Program:
    """Parse `source` as a program in the evaluated subset."""
    tree = _parse(source, "program", filename)
    return _SyntaxTransformer().transform(tree)


def _parse(source: str, start: str, filename: str | None):
    try:
        return _LARK.parse(source.lower(), start=start)
    except UnexpectedInput as e:
        raise _parse_error(e, source, filename) from None


# ================= Error reporting =================

_KEYWORDS = {
    "DEFINE", "LAMBDA", "IF", "LET", "AND", "OR", "NOT", "CAR",
    "CDR", "CONS", "NIL", "EQ", "ATOM", "INTP", "QUOTE", "QUASIQUOTE",
    "UNQUOTE", "BEGIN",
}  # fmt: skip

_PREFIXES = {"QUOTE_MARK", "BACKQUOTE", "COMMA"}
_DATUM_STARTERS = {"LPAR", "INTEGER", "BOOLEAN", "WORD", "OPERATOR", "DOT"} | _PREFIXES

# Terminals that can begin a datum / an expression. When all of them are
# expected, the error says "a datum" / "an expression" instead of listing each.
# Datums come first: they're a superset, and appear in both grammars (quote).
# Quasiquote templates are datums whose atoms can't be quasiquote/unquote.
_STARTERS = (
    ("a datum", _DATUM_STARTERS | _KEYWORDS),
    ("a datum", _DATUM_STARTERS | _KEYWORDS - {"QUASIQUOTE", "UNQUOTE"}),
    (
        "an expression",
        {"LPAR", "INTEGER", "BOOLEAN", "WORD", "NIL", "QUOTE_MARK", "BACKQUOTE"},
    ),
)

_TERMINAL_NAMES = {
    "INTEGER": "an integer",
    "BOOLEAN": "a boolean",
    "WORD": "an identifier",
    "OPERATOR": "an operator",
    "$END": "end of input",
}


def _parse_error(e: UnexpectedInput, source: str, filename: str | None) -> ParseError:
    match e:
        case UnexpectedCharacters():
            line, column = e.line, e.column
            message = f"unexpected character {source[e.pos_in_stream]!r}"
        case UnexpectedToken() if e.token.type != "$END":
            line, column = e.line, e.column
            text = source[e.token.start_pos : e.token.end_pos]
            message = f"unexpected {text!r}, expected {_describe(e.expected)}"
        case UnexpectedToken() | UnexpectedEOF():
            # Point just past the last non-blank character.
            before = source.rstrip()
            line = before.count("\n") + 1
            column = len(before) - (before.rfind("\n") + 1) + 1
            message = f"unexpected end of input, expected {_describe(e.expected)}"
        case _:
            line, column = e.line, e.column
            message = str(e)
    return ParseError(message, line, column, _context(source, line, column), filename)


def _describe(expected: set[str] | list[str]) -> str:
    names = set(expected)
    parts = []
    for noun, starters in _STARTERS:
        if starters <= names:
            names -= starters
            parts.append(noun)
    parts += sorted(_terminal_name(name) for name in names)
    if len(parts) == 1:
        return parts[0]
    return f"{', '.join(parts[:-1])} or {parts[-1]}"


def _terminal_name(name: str) -> str:
    if name in _TERMINAL_NAMES:
        return _TERMINAL_NAMES[name]
    pattern = _LARK.get_terminal(name).pattern
    return repr(pattern.value) if isinstance(pattern, PatternStr) else name


def _context(source: str, line: int, column: int) -> str:
    lines = source.splitlines() or [""]
    text = lines[min(line, len(lines)) - 1]
    # Keep tabs so the caret lines up with tab-indented source.
    pad = "".join(c if c == "\t" else " " for c in text[: column - 1])
    return f"    {text}\n    {pad}^"


# ================= Tree -> data =================


@v_args(inline=True)
class _ReaderTransformer(Transformer):
    def reader(self, *datums: Datum) -> tuple[Datum, ...]:
        return datums

    def list(self, *datums: Datum) -> tuple[Datum, ...]:
        return datums

    def quoted(self, datum: Datum) -> tuple[Datum, ...]:
        return (Symbol("quote"), datum)

    def quasiquoted(self, datum: Datum) -> tuple[Datum, ...]:
        return (Symbol("quasiquote"), datum)

    def unquoted(self, datum: Datum) -> tuple[Datum, ...]:
        return (Symbol("unquote"), datum)

    def atom(self, token: Token) -> Datum:
        match token.type:
            case "INTEGER":
                return int(token)
            case "BOOLEAN":
                return token == "#t"
            case _:
                return Symbol(token)


# Inherits the datum rules, which appear inside quote.
@v_args(inline=True)
class _SyntaxTransformer(_ReaderTransformer):
    def program(self, *forms):
        return Program(forms)

    def define(self, name: Token, value):
        return Define(str(name), value)

    def define_function(self, name: Token, params: tuple[str, ...], body):
        return Define(str(name), Lambda(params, body))

    def define_block(self, *definitions):
        return DefineBlock(definitions)

    def integer(self, token: Token):
        return Integer(int(token))

    def boolean(self, token: Token):
        return Boolean(token == "#t")

    def nil(self):
        return Nil()

    def variable(self, token: Token):
        return Variable(str(token))

    def quote(self, datum: Datum):
        return Quote(datum)

    def quasiquote(self, template: Template):
        return Quasiquote(template)

    template_atom = _ReaderTransformer.atom

    def template_list(self, *templates: Template) -> tuple[Template, ...]:
        return templates

    def template_quoted(self, template: Template) -> tuple[Template, ...]:
        return (Symbol("quote"), template)

    def template_quasiquoted(self, datum: Datum) -> tuple[Datum, ...]:
        return (Symbol("quasiquote"), datum)

    def unquote(self, expr):
        return Unquote(expr)

    def lambda_(self, params: tuple[str, ...], body):
        return Lambda(params, body)

    def conditional(self, test, then, orelse):
        return If(test, then, orelse)

    def sequence(self, *forms):
        return Sequence(forms)

    def primitive_call(self, op: Token, *args):
        return PrimitiveCall(str(op), args)

    def application(self, func, *args):
        return Application(func, args)

    def formals(self, *names: Token) -> tuple[str, ...]:
        return tuple(str(name) for name in names)

```

To demonstrate that the grammar is correct, we can produce an example:
quarto-executable-code-5450563D

```python
#| code-fold: false

from pprint import pprint

pprint(parse("(define (foo bar) bar)"))
pprint(parse("(+ 1 2)"))
```

## Encoding {#sec-enc}

### Basics of Language Encoding

- Explain how we can encode $\mathcal{L}_\text{fruit}$

### Cleanup Memory

- Cleanup memory and VSA wrapper

### Associative Memory

- Associative memory and VSA wrapper

### Lists and Abstract Syntax

- Talk about `VSAList`

### Lambdas

### Numbers {#sec-numbers}

#### Peano Integer {#sec-peano-int}

#### aTRHC Integers {#sec-atrhc-int}

#### Quoting and quasiquotation

## Interpretation {#sec-interp}

### Semantics {#sec-semantics}

### The Semantics of LISP$_\text{Plate}$
````